## Synthetic Data, Panels

In [5]:
# -*- coding: utf-8 -*-
"""
Synthetic wave + time-varying planar bias animation.

For each requested keogram angle (0 and 90 deg), this script saves a separate
4 x 3 animation:

Row 1: Large 5000 x 5000 km FOV, lower spatial resolution
    left   = wave only
    center = bias only
    right  = wave + bias
    A box marks the central 500 x 500 km FOV used below.

Row 2: Central 500 x 500 km FOV at 1 km resolution
    left   = wave only
    center = bias only
    right  = wave + bias
    Each panel includes an arrow showing the plane-fit slope vector
    for the current frame.

Row 3: Current keogram up to the current frame
    left   = wave only
    center = bias only
    right  = wave + bias

Row 4: Time profiles at 0, 125, and 250 km along the current keogram cut
    left   = wave only
    center = bias only
    right  = wave + bias

Two separate static diagnostic figures are also saved:
    1. Plane-fit slope vectors over time. Points show every frame, with very
       faint arrows from the origin; color represents time.
    2. A 3-panel bias comparison showing the fitted wave+bias slopes, the
       calculated bias, and the actual imposed bias.

Requested keogram loop:
    0 deg  = E-W cut through y = 0
    90 deg = N-S cut through x = 0

Bias specification:
    at   0 min: x slope = -0.020 amp/100 km, y slope = +0.005 amp/100 km
    at 180 min: x slope = +0.020 amp/100 km, y slope = +0.005 amp/100 km
    x varies linearly from 0 to 180 min and is then held constant through
    the end of the 240-min simulation; y stays constant.

Calculated-bias diagnostic:
    - Average all wave+bias fitted slope points over 0-120 min; assign that
      average to 30 min.
    - Average all wave+bias fitted slope points over 180-300 min; assign that
      average to 210 min.
    - The calculated bias from 30 to 210 min is the straight line connecting
      those two average slope vectors.

Coordinate convention:
    +x = right/east
    +y = down/south
    0 deg = +x/east
    +90 deg = +y/south

At DT_SECONDS = 120 s, including both 0 and 240 min gives 121 frames.
"""

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle


# ==================================================
# Output
# ==================================================
OUT_DIR = Path("synthetic_wave_bias_animation_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_BASENAME = "synthetic_wave_bias_0_to_240min"

# Animation settings
FPS = 8
ANIMATION_DPI = 110

# "mp4" is preferred. If ffmpeg is unavailable, the script automatically
# falls back to GIF using Pillow.
PREFERRED_ANIMATION_FORMAT = "mp4"


# ==================================================
# Spatial grids
# ==================================================
# Central high-resolution FOV
SMALL_FIELD_SIZE_KM = 500.0
SMALL_DX_KM = 1.0
SMALL_DY_KM = 1.0

# Large overview FOV. 20-km sampling gives 251 x 251 pixels over 5000 km.
BIG_FIELD_SIZE_KM = 5000.0
BIG_DX_KM = 20.0
BIG_DY_KM = 20.0


# ==================================================
# Temporal grid
# ==================================================
DT_SECONDS = 120
TOTAL_TIME_MIN = 300.0

times_s = np.arange(
    0.0,
    TOTAL_TIME_MIN * 60.0 + DT_SECONDS,
    DT_SECONDS,
    dtype=np.float64,
)
times_min = times_s / 60.0
NT = len(times_min)


# ==================================================
# Wave parameters
# ==================================================
WAVELENGTH_KM = 4050.0
GAUSSIAN_WIDTH_KM = 7200.0  # Gaussian sigma
MAX_AMPLITUDE = 0.06

PROPAGATION_ANGLE_START_TIME_MIN = 0.0
PROPAGATION_ANGLE_END_TIME_MIN = 300.0
PROPAGATION_ANGLE_START_DEG = -80.0
PROPAGATION_ANGLE_END_DEG = -80.0

WAVE_SPEED_M_S = -450.0
WAVE_TIME_SHIFT_MIN = 90.0


# ==================================================
# Bias specification
# ==================================================
# Units: perturbation amplitude per 100 km.
BIAS_X_START_PER_100KM = -0.03
BIAS_X_END_PER_100KM = +0.045
BIAS_Y_START_PER_100KM = +0.008
BIAS_Y_END_PER_100KM = +0.004

# Keep the original bias schedule: it reaches its end value at 180 min,
# then remains at that value while the simulation continues to 240 min.
BIAS_END_TIME_MIN = 300.0

# Calculated-bias endpoint windows.
CALC_BIAS_FIRST_CENTER_MIN = 60.0
CALC_BIAS_SECOND_CENTER_MIN = 240.0
CALC_BIAS_HALF_WINDOW_MIN = 60.0


# ==================================================
# Keogram / profile settings
# ==================================================
# Make one complete animation for each of these directions.
KEOGRAM_ANGLES_DEG = [0.0, 90.0]

KEOGRAM_WIDTH_PIXELS = 7
PROFILE_DISTANCES_KM = [0.0, 125.0, 250.0]


# ==================================================
# Plot settings
# ==================================================
CMAP = "RdBu_r"

# Fixed display limits keep colors stable from frame to frame.
TOP_WAVE_VMAX = MAX_AMPLITUDE

# For the large FOV, the largest possible planar bias occurs near a corner.
# abs(bx)*25 + abs(by)*25 because 2500 km = 25 * 100 km.
TOP_BIAS_VMAX = (
    max(abs(BIAS_X_START_PER_100KM), abs(BIAS_X_END_PER_100KM))
    * (BIG_FIELD_SIZE_KM / 2.0 / 100.0)
    + max(abs(BIAS_Y_START_PER_100KM), abs(BIAS_Y_END_PER_100KM))
    * (BIG_FIELD_SIZE_KM / 2.0 / 100.0)
)
TOP_COMBINED_VMAX = TOP_BIAS_VMAX + MAX_AMPLITUDE

# Use one common range for the three 500 x 500 km panels and keograms.
SMALL_COMMON_VMAX = 0.15
PROFILE_YLIM = (-SMALL_COMMON_VMAX, SMALL_COMMON_VMAX)

# Arrow scaling in the image panels. The largest plane-fit slope seen over
# the animation is mapped to this maximum arrow length in km.
MAX_ARROW_LENGTH_KM = 115.0

# Static slope-vector plot.
SLOPE_VECTOR_CMAP = "viridis"
SLOPE_VECTOR_DPI = 250


# ==================================================
# Coordinate grids
# ==================================================
def centered_grid(field_size_km, dx_km, dy_km):
    nx = int(round(field_size_km / dx_km)) + 1
    ny = int(round(field_size_km / dy_km)) + 1

    x = np.linspace(-field_size_km / 2.0, field_size_km / 2.0, nx)
    y = np.linspace(-field_size_km / 2.0, field_size_km / 2.0, ny)
    X, Y = np.meshgrid(x, y)
    return x, y, X, Y


x_small, y_small, X_SMALL, Y_SMALL = centered_grid(
    SMALL_FIELD_SIZE_KM,
    SMALL_DX_KM,
    SMALL_DY_KM,
)

x_big, y_big, X_BIG, Y_BIG = centered_grid(
    BIG_FIELD_SIZE_KM,
    BIG_DX_KM,
    BIG_DY_KM,
)

NY_SMALL, NX_SMALL = X_SMALL.shape


# ==================================================
# Time-varying wave state
# ==================================================
def propagation_angle_deg_at_time(time_min):
    """Prescribed propagation angle."""
    t = np.asarray(time_min, dtype=np.float64)

    fraction = (
        t - PROPAGATION_ANGLE_START_TIME_MIN
    ) / (
        PROPAGATION_ANGLE_END_TIME_MIN
        - PROPAGATION_ANGLE_START_TIME_MIN
    )
    fraction = np.clip(fraction, 0.0, 1.0)

    return (
        PROPAGATION_ANGLE_START_DEG
        + fraction
        * (PROPAGATION_ANGLE_END_DEG - PROPAGATION_ANGLE_START_DEG)
    )


propagation_angles_deg = propagation_angle_deg_at_time(times_min)
propagation_angles_rad = np.deg2rad(propagation_angles_deg)

wavenumber_rad_per_km = 2.0 * np.pi / WAVELENGTH_KM
wave_speed_km_s = WAVE_SPEED_M_S / 1000.0

# Integrate the moving packet-center trajectory.
velocity_x_km_s = wave_speed_km_s * np.cos(propagation_angles_rad)
velocity_y_km_s = wave_speed_km_s * np.sin(propagation_angles_rad)

packet_center_x_km = np.zeros(NT, dtype=np.float64)
packet_center_y_km = np.zeros(NT, dtype=np.float64)

if NT > 1:
    dt_integration_s = np.diff(times_s)
    packet_center_x_km[1:] = np.cumsum(
        0.5
        * (velocity_x_km_s[:-1] + velocity_x_km_s[1:])
        * dt_integration_s
    )
    packet_center_y_km[1:] = np.cumsum(
        0.5
        * (velocity_y_km_s[:-1] + velocity_y_km_s[1:])
        * dt_integration_s
    )

# Carrier-wave temporal phase.
angular_frequency_rad_s = wavenumber_rad_per_km * wave_speed_km_s

temporal_phase_rad = angular_frequency_rad_s * times_s
phase_shift_rad = (
    angular_frequency_rad_s
    * WAVE_TIME_SHIFT_MIN
    * 60.0
)
temporal_phase_rad = temporal_phase_rad + phase_shift_rad


# ==================================================
# Time-varying bias state
# ==================================================
bias_fraction = np.clip(
    times_min / BIAS_END_TIME_MIN,
    0.0,
    1.0,
)

bias_x_per_100km = (
    BIAS_X_START_PER_100KM
    + bias_fraction
    * (BIAS_X_END_PER_100KM - BIAS_X_START_PER_100KM)
)

bias_y_per_100km = (
    BIAS_Y_START_PER_100KM
    + bias_fraction
    * (BIAS_Y_END_PER_100KM - BIAS_Y_START_PER_100KM)
)

bias_magnitude_per_100km = np.hypot(
    bias_x_per_100km,
    bias_y_per_100km,
)

bias_angle_deg = np.degrees(
    np.arctan2(bias_y_per_100km, bias_x_per_100km)
)
bias_angle_deg = (bias_angle_deg + 180.0) % 360.0 - 180.0


# ==================================================
# Frame generators
# ==================================================
def make_wave_frame(frame_index, X, Y):
    """Generate one wave frame on an arbitrary centered X/Y grid."""
    theta = propagation_angles_rad[frame_index]
    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)

    relative_x_km = X - packet_center_x_km[frame_index]
    relative_y_km = Y - packet_center_y_km[frame_index]

    parallel_relative_km = (
        relative_x_km * cos_theta
        + relative_y_km * sin_theta
    )

    gaussian_envelope = np.exp(
        -0.5 * (parallel_relative_km / GAUSSIAN_WIDTH_KM) ** 2
    )

    spatial_phase_rad = wavenumber_rad_per_km * (
        X * cos_theta + Y * sin_theta
    )

    wave_phase_rad = spatial_phase_rad - temporal_phase_rad[frame_index]

    return (
        MAX_AMPLITUDE
        * gaussian_envelope
        * np.cos(wave_phase_rad)
    ).astype(np.float32)


def make_bias_frame(frame_index, X, Y):
    """Generate one planar-bias frame on an arbitrary centered X/Y grid."""
    bx = bias_x_per_100km[frame_index]
    by = bias_y_per_100km[frame_index]

    return (
        bx * X / 100.0
        + by * Y / 100.0
    ).astype(np.float32)


def make_three_frames(frame_index, X, Y):
    """Return wave, bias, and wave+bias for one time/frame."""
    wave = make_wave_frame(frame_index, X, Y)
    bias = make_bias_frame(frame_index, X, Y)
    combined = wave + bias
    return wave, bias, combined


# ==================================================
# Plane-fit slope helpers
# ==================================================
# For a centered rectangular grid, x and y are orthogonal and zero-mean.
X_SMALL_64 = X_SMALL.astype(np.float64)
Y_SMALL_64 = Y_SMALL.astype(np.float64)
SUM_X2 = np.sum(X_SMALL_64 ** 2)
SUM_Y2 = np.sum(Y_SMALL_64 ** 2)


def fit_plane_slope(frame):
    """
    Fit z = intercept + slope_x*x + slope_y*y.

    Returns slope_x and slope_y in amplitude per 100 km.
    """
    z = np.asarray(frame, dtype=np.float64)

    slope_x_per_km = np.sum(X_SMALL_64 * z) / SUM_X2
    slope_y_per_km = np.sum(Y_SMALL_64 * z) / SUM_Y2

    return (
        100.0 * slope_x_per_km,
        100.0 * slope_y_per_km,
    )


# ==================================================
# Directional keogram helper
# ==================================================
def extract_directional_cut(frame, angle_deg, width_pixels=7):
    """
    Extract a center cut at 0 or 90 degrees.

    0 deg:
        E-W cut through y = 0.
        Average a horizontal strip across y.
        Returned distance coordinate is x.

    90 deg:
        N-S cut through x = 0.
        Average a vertical strip across x.
        Returned distance coordinate is y.

    Returns
    -------
    distance_km : 1-D ndarray
    cut : 1-D ndarray
    """
    angle = float(angle_deg) % 180.0
    half_width = width_pixels // 2

    if np.isclose(angle, 0.0):
        center_row = (frame.shape[0] - 1) // 2
        row_start = max(0, center_row - half_width)
        row_stop = min(frame.shape[0], center_row + half_width + 1)

        cut = np.nanmean(frame[row_start:row_stop, :], axis=0)
        return x_small, cut

    if np.isclose(angle, 90.0):
        center_col = (frame.shape[1] - 1) // 2
        col_start = max(0, center_col - half_width)
        col_stop = min(frame.shape[1], center_col + half_width + 1)

        cut = np.nanmean(frame[:, col_start:col_stop], axis=1)
        return y_small, cut

    raise ValueError(
        "This script currently supports only 0 and 90 degree cuts. "
        f"Received {angle_deg} deg."
    )


# ==================================================
# Precompute small-FOV diagnostics
# ==================================================
variant_names = ["Wave only", "Bias only", "Wave + bias"]

N_ANGLES = len(KEOGRAM_ANGLES_DEG)

# The 0 and 90 deg cuts have the same number of samples here because
# SMALL_DX_KM == SMALL_DY_KM and the FOV is square.
N_CUT = NX_SMALL

keograms = np.empty(
    (N_ANGLES, 3, NT, N_CUT),
    dtype=np.float32,
)

plane_slope_x = np.empty((3, NT), dtype=np.float64)
plane_slope_y = np.empty((3, NT), dtype=np.float64)

print(f"Precomputing diagnostics for {NT} frames...")

for it in range(NT):
    small_frames = make_three_frames(it, X_SMALL, Y_SMALL)

    for variant_index, frame in enumerate(small_frames):
        sx, sy = fit_plane_slope(frame)
        plane_slope_x[variant_index, it] = sx
        plane_slope_y[variant_index, it] = sy

        for angle_index, angle_deg in enumerate(KEOGRAM_ANGLES_DEG):
            _, cut = extract_directional_cut(
                frame,
                angle_deg=angle_deg,
                width_pixels=KEOGRAM_WIDTH_PIXELS,
            )
            keograms[angle_index, variant_index, it] = cut


plane_slope_magnitude = np.hypot(plane_slope_x, plane_slope_y)
max_plane_slope = np.nanmax(plane_slope_magnitude)

if not np.isfinite(max_plane_slope) or max_plane_slope <= 0.0:
    max_plane_slope = 1.0


# ==================================================
# Calculate bias from the fitted wave+bias slope
# ==================================================
# Variant index 2 is "Wave + bias".
combined_slope_x = plane_slope_x[2]
combined_slope_y = plane_slope_y[2]

first_window_start_min = (
    CALC_BIAS_FIRST_CENTER_MIN - CALC_BIAS_HALF_WINDOW_MIN
)
first_window_end_min = (
    CALC_BIAS_FIRST_CENTER_MIN + CALC_BIAS_HALF_WINDOW_MIN
)
second_window_start_min = (
    CALC_BIAS_SECOND_CENTER_MIN - CALC_BIAS_HALF_WINDOW_MIN
)
second_window_end_min = (
    CALC_BIAS_SECOND_CENTER_MIN + CALC_BIAS_HALF_WINDOW_MIN
)

first_window_mask = (
    (times_min >= first_window_start_min)
    & (times_min <= first_window_end_min)
)
second_window_mask = (
    (times_min >= second_window_start_min)
    & (times_min <= second_window_end_min)
)

calc_bias_anchor_x = np.array(
    [
        np.nanmean(combined_slope_x[first_window_mask]),
        np.nanmean(combined_slope_x[second_window_mask]),
    ],
    dtype=np.float64,
)
calc_bias_anchor_y = np.array(
    [
        np.nanmean(combined_slope_y[first_window_mask]),
        np.nanmean(combined_slope_y[second_window_mask]),
    ],
    dtype=np.float64,
)
calc_bias_anchor_times_min = np.array(
    [
        CALC_BIAS_FIRST_CENTER_MIN,
        CALC_BIAS_SECOND_CENTER_MIN,
    ],
    dtype=np.float64,
)

# Only define the calculated bias on the requested 30-210 min interval.
calc_bias_x_per_100km = np.full(NT, np.nan, dtype=np.float64)
calc_bias_y_per_100km = np.full(NT, np.nan, dtype=np.float64)

calc_bias_interval_mask = (
    (times_min >= CALC_BIAS_FIRST_CENTER_MIN)
    & (times_min <= CALC_BIAS_SECOND_CENTER_MIN)
)

calc_fraction = (
    times_min[calc_bias_interval_mask] - CALC_BIAS_FIRST_CENTER_MIN
) / (
    CALC_BIAS_SECOND_CENTER_MIN - CALC_BIAS_FIRST_CENTER_MIN
)

calc_bias_x_per_100km[calc_bias_interval_mask] = (
    calc_bias_anchor_x[0]
    + calc_fraction
    * (calc_bias_anchor_x[1] - calc_bias_anchor_x[0])
)
calc_bias_y_per_100km[calc_bias_interval_mask] = (
    calc_bias_anchor_y[0]
    + calc_fraction
    * (calc_bias_anchor_y[1] - calc_bias_anchor_y[0])
)

print(
    "Calculated-bias anchor at "
    f"{CALC_BIAS_FIRST_CENTER_MIN:.0f} min "
    f"(average {first_window_start_min:.0f}-"
    f"{first_window_end_min:.0f} min): "
    f"x={calc_bias_anchor_x[0]:+.6f}, "
    f"y={calc_bias_anchor_y[0]:+.6f} amp/100 km"
)
print(
    "Calculated-bias anchor at "
    f"{CALC_BIAS_SECOND_CENTER_MIN:.0f} min "
    f"(average {second_window_start_min:.0f}-"
    f"{second_window_end_min:.0f} min): "
    f"x={calc_bias_anchor_x[1]:+.6f}, "
    f"y={calc_bias_anchor_y[1]:+.6f} amp/100 km"
)


# ==================================================
# Profiles for each cut angle
# ==================================================
# distances_by_angle has shape [N_ANGLES, N_CUT].
distances_by_angle = np.empty((N_ANGLES, N_CUT), dtype=np.float64)

# profiles shape:
# [angle, variant, requested profile distance, time]
profiles = np.empty(
    (
        N_ANGLES,
        3,
        len(PROFILE_DISTANCES_KM),
        NT,
    ),
    dtype=np.float32,
)

profile_actual_distances = np.empty(
    (N_ANGLES, len(PROFILE_DISTANCES_KM)),
    dtype=np.float64,
)

for angle_index, angle_deg in enumerate(KEOGRAM_ANGLES_DEG):
    if np.isclose(angle_deg % 180.0, 0.0):
        distance_axis = x_small
    elif np.isclose(angle_deg % 180.0, 90.0):
        distance_axis = y_small
    else:
        raise ValueError("Only 0 and 90 degree cuts are supported.")

    distances_by_angle[angle_index] = distance_axis

    for profile_index, requested_distance in enumerate(PROFILE_DISTANCES_KM):
        distance_index = int(
            np.argmin(np.abs(distance_axis - requested_distance))
        )
        profile_actual_distances[angle_index, profile_index] = (
            distance_axis[distance_index]
        )

        for variant_index in range(3):
            profiles[
                angle_index,
                variant_index,
                profile_index,
            ] = keograms[
                angle_index,
                variant_index,
                :,
                distance_index,
            ]


# ==================================================
# Save useful diagnostic arrays
# ==================================================
np.savez_compressed(
    OUT_DIR / "animation_diagnostics.npz",
    times_min=times_min,
    x_small_km=x_small,
    y_small_km=y_small,
    keogram_angles_deg=np.asarray(KEOGRAM_ANGLES_DEG, dtype=float),
    keogram_distances_km=distances_by_angle,
    propagation_angles_deg=propagation_angles_deg,
    packet_center_x_km=packet_center_x_km,
    packet_center_y_km=packet_center_y_km,
    bias_x_per_100km=bias_x_per_100km,
    bias_y_per_100km=bias_y_per_100km,
    bias_angle_deg=bias_angle_deg,
    bias_magnitude_per_100km=bias_magnitude_per_100km,
    plane_slope_x_per_100km=plane_slope_x,
    plane_slope_y_per_100km=plane_slope_y,
    calculated_bias_anchor_times_min=calc_bias_anchor_times_min,
    calculated_bias_anchor_x_per_100km=calc_bias_anchor_x,
    calculated_bias_anchor_y_per_100km=calc_bias_anchor_y,
    calculated_bias_x_per_100km=calc_bias_x_per_100km,
    calculated_bias_y_per_100km=calc_bias_y_per_100km,
    keograms=keograms,
    profile_requested_distances_km=np.asarray(PROFILE_DISTANCES_KM),
    profile_actual_distances_km=profile_actual_distances,
    profiles=profiles,
)


# ==================================================
# Separate slope-arrow-over-time plot
# ==================================================
def save_slope_vector_plot():
    """
    Plot every frame's plane-fit slope tip as a point.

    Very faint arrows from the origin are retained only to make the vector
    direction easy to see. Point/arrow color represents time.
    """
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18.5, 5.3),
        sharex=True,
        sharey=True,
    )

    norm = Normalize(vmin=times_min[0], vmax=times_min[-1])
    cmap = plt.get_cmap(SLOPE_VECTOR_CMAP)

    component_limit = np.nanmax(
        np.abs(
            np.concatenate(
                [
                    plane_slope_x.ravel(),
                    plane_slope_y.ravel(),
                ]
            )
        )
    )

    if not np.isfinite(component_limit) or component_limit <= 0.0:
        component_limit = 1.0

    component_limit *= 1.10

    point_size = 20

    for col, ax in enumerate(axes):
        # Very faint arrows so the dense plot is not dominated by arrowheads.
        ax.quiver(
            np.zeros(NT),
            np.zeros(NT),
            plane_slope_x[col],
            plane_slope_y[col],
            times_min,
            cmap=cmap,
            norm=norm,
            angles="xy",
            scale_units="xy",
            scale=1.0,
            width=0.0020,
            headwidth=3.0,
            headlength=4.0,
            headaxislength=3.5,
            alpha=0.12,
            zorder=1,
        )

        # Main representation: slope-vector tips as time-colored points.
        ax.scatter(
            plane_slope_x[col],
            plane_slope_y[col],
            c=times_min,
            cmap=cmap,
            norm=norm,
            s=point_size,
            linewidths=0.0,
            alpha=0.92,
            zorder=3,
        )

        # Thin trajectory through the slope-vector tips.
        ax.plot(
            plane_slope_x[col],
            plane_slope_y[col],
            linewidth=0.75,
            alpha=0.28,
            color="0.25",
            zorder=2,
        )

        # Mark the first and last points.
        ax.scatter(
            [plane_slope_x[col, 0]],
            [plane_slope_y[col, 0]],
            s=46,
            facecolors="none",
            edgecolors="black",
            linewidths=1.1,
            zorder=5,
            label="0 min",
        )
        ax.scatter(
            [plane_slope_x[col, -1]],
            [plane_slope_y[col, -1]],
            s=44,
            marker="x",
            color="black",
            linewidths=1.2,
            zorder=5,
            label=f"{times_min[-1]:.0f} min",
        )

        ax.axhline(0.0, color="0.6", linewidth=0.8)
        ax.axvline(0.0, color="0.6", linewidth=0.8)

        ax.set_xlim(-component_limit, component_limit)

        # +y is downward in the image coordinate convention.
        ax.set_ylim(component_limit, -component_limit)

        ax.set_aspect("equal", adjustable="box")
        ax.grid(True, alpha=0.22)
        ax.set_title(variant_names[col])
        ax.set_xlabel("x slope (amp/100 km)")

        if col == 0:
            ax.set_ylabel("y slope (amp/100 km; + downward)")

        ax.legend(loc="best", fontsize=8)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    # Reserve a dedicated strip on the right so the colorbar cannot overlap
    # the third subplot.
    cbar_ax = fig.add_axes([0.935, 0.18, 0.012, 0.66])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label("Time (min)")

    fig.suptitle(
        "Plane-fit slope vectors over time",
        fontsize=14,
    )

    fig.subplots_adjust(
        top=0.86,
        bottom=0.14,
        left=0.06,
        right=0.90,
        wspace=0.20,
    )

    output_path = OUT_DIR / "slope_vectors_over_time.png"
    fig.savefig(
        output_path,
        dpi=SLOPE_VECTOR_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

    print("Slope-vector plot:", output_path)
    return output_path


# ==================================================
# Calculated-bias comparison plot
# ==================================================
def save_calculated_bias_plot():
    """
    Save a 3-panel slope-vector-space comparison:

    1. Plane-fit slope of wave + actual bias.
    2. Calculated bias: straight line from the 0-120 min average (placed at
       60 min) to the 180-300 min average (placed at 240 min).
    3. Actual imposed bias for comparison.
    """
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18.5, 5.3),
        sharex=True,
        sharey=True,
    )

    norm = Normalize(vmin=times_min[0], vmax=times_min[-1])
    cmap = plt.get_cmap(SLOPE_VECTOR_CMAP)

    finite_calc = np.isfinite(calc_bias_x_per_100km) & np.isfinite(
        calc_bias_y_per_100km
    )

    all_x = np.concatenate(
        [
            combined_slope_x,
            calc_bias_x_per_100km[finite_calc],
            bias_x_per_100km,
        ]
    )
    all_y = np.concatenate(
        [
            combined_slope_y,
            calc_bias_y_per_100km[finite_calc],
            bias_y_per_100km,
        ]
    )

    component_limit = np.nanmax(np.abs(np.concatenate([all_x, all_y])))
    if not np.isfinite(component_limit) or component_limit <= 0.0:
        component_limit = 1.0
    component_limit *= 1.12

    def draw_vector_points(
        ax,
        x_values,
        y_values,
        color_times,
        point_alpha=0.92,
        arrow_alpha=0.10,
    ):
        ax.quiver(
            np.zeros_like(x_values),
            np.zeros_like(y_values),
            x_values,
            y_values,
            color_times,
            cmap=cmap,
            norm=norm,
            angles="xy",
            scale_units="xy",
            scale=1.0,
            width=0.0020,
            headwidth=3.0,
            headlength=4.0,
            headaxislength=3.5,
            alpha=arrow_alpha,
            zorder=1,
        )
        ax.scatter(
            x_values,
            y_values,
            c=color_times,
            cmap=cmap,
            norm=norm,
            s=22,
            linewidths=0.0,
            alpha=point_alpha,
            zorder=3,
        )
        ax.plot(
            x_values,
            y_values,
            color="0.25",
            linewidth=0.8,
            alpha=0.30,
            zorder=2,
        )

    # Panel 1: measured/fitted slope of wave + bias.
    draw_vector_points(
        axes[0],
        combined_slope_x,
        combined_slope_y,
        times_min,
        point_alpha=0.92,
        arrow_alpha=0.10,
    )
    axes[0].set_title("Wave + bias plane-fit slope")

    # Show which points enter the two endpoint averages.
    axes[0].scatter(
        combined_slope_x[first_window_mask],
        combined_slope_y[first_window_mask],
        s=52,
        facecolors="none",
        edgecolors="black",
        linewidths=0.7,
        alpha=0.45,
        zorder=4,
        label="0-120 and 180-300 min windows",
    )
    axes[0].scatter(
        combined_slope_x[second_window_mask],
        combined_slope_y[second_window_mask],
        s=52,
        facecolors="none",
        edgecolors="black",
        linewidths=0.7,
        alpha=0.45,
        zorder=4,
    )
    axes[0].legend(loc="best", fontsize=8)

    # Panel 2: calculated bias only between 30 and 210 min.
    calc_times = times_min[finite_calc]
    calc_x = calc_bias_x_per_100km[finite_calc]
    calc_y = calc_bias_y_per_100km[finite_calc]

    draw_vector_points(
        axes[1],
        calc_x,
        calc_y,
        calc_times,
        point_alpha=0.95,
        arrow_alpha=0.10,
    )
    axes[1].scatter(
        calc_bias_anchor_x,
        calc_bias_anchor_y,
        c=calc_bias_anchor_times_min,
        cmap=cmap,
        norm=norm,
        s=90,
        edgecolors="black",
        linewidths=1.0,
        zorder=5,
    )
    axes[1].set_title("Calculated bias (60-240 min)")
    axes[1].text(
        0.03,
        0.97,
        "60 min = mean of 0-120 min\n"
        "240 min = mean of 180-300 min",
        transform=axes[1].transAxes,
        ha="left",
        va="top",
        fontsize=8.5,
        bbox=dict(
            facecolor="white",
            alpha=0.78,
            edgecolor="none",
        ),
        zorder=6,
    )

    # Panel 3: actual imposed bias.
    draw_vector_points(
        axes[2],
        bias_x_per_100km,
        bias_y_per_100km,
        times_min,
        point_alpha=0.92,
        arrow_alpha=0.10,
    )
    axes[2].set_title("Actual bias")

    for col, ax in enumerate(axes):
        ax.axhline(0.0, color="0.6", linewidth=0.8)
        ax.axvline(0.0, color="0.6", linewidth=0.8)
        ax.set_xlim(-component_limit, component_limit)
        ax.set_ylim(component_limit, -component_limit)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(True, alpha=0.22)
        ax.set_xlabel("x slope (amp/100 km)")

        if col == 0:
            ax.set_ylabel("y slope (amp/100 km; + downward)")

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar_ax = fig.add_axes([0.935, 0.18, 0.012, 0.66])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label("Time (min)")

    fig.suptitle(
        "Calculated bias from wave+bias plane-fit slopes",
        fontsize=14,
    )

    fig.subplots_adjust(
        top=0.86,
        bottom=0.14,
        left=0.06,
        right=0.90,
        wspace=0.20,
    )

    output_path = OUT_DIR / "calculated_bias_comparison.png"
    fig.savefig(
        output_path,
        dpi=SLOPE_VECTOR_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

    print("Calculated-bias plot:", output_path)
    return output_path


slope_plot_path = save_slope_vector_plot()
calculated_bias_plot_path = save_calculated_bias_plot()


# ==================================================
# Animation builder
# ==================================================
def build_animation(angle_index, keogram_angle_deg):
    """
    Build one full animation for one keogram/profile direction.
    """
    distance_axis = distances_by_angle[angle_index]
    angle_keograms = keograms[angle_index]
    angle_profiles = profiles[angle_index]
    angle_profile_distances = profile_actual_distances[angle_index]

    fig, axes = plt.subplots(
        4,
        3,
        figsize=(18.5, 15.5),
        gridspec_kw={
            "height_ratios": [1.25, 1.25, 0.95, 0.95],
            "hspace": 0.34,
            "wspace": 0.24,
        },
    )

    fig.subplots_adjust(
        top=0.94,
        bottom=0.06,
        left=0.055,
        right=0.91,
    )

    # --------------------------------------------------
    # Row 1: big FOV images
    # --------------------------------------------------
    initial_big_frames = make_three_frames(0, X_BIG, Y_BIG)

    top_vmax = [TOP_WAVE_VMAX, TOP_BIAS_VMAX, TOP_COMBINED_VMAX]
    top_images = []

    for col in range(3):
        ax = axes[0, col]

        image = ax.imshow(
            initial_big_frames[col],
            cmap=CMAP,
            vmin=-top_vmax[col],
            vmax=top_vmax[col],
            origin="upper",
            extent=[
                -BIG_FIELD_SIZE_KM / 2.0,
                BIG_FIELD_SIZE_KM / 2.0,
                BIG_FIELD_SIZE_KM / 2.0,
                -BIG_FIELD_SIZE_KM / 2.0,
            ],
            interpolation="nearest",
            aspect="equal",
        )
        top_images.append(image)

        # Box marking the central 500 x 500 km FOV.
        box = Rectangle(
            (
                -SMALL_FIELD_SIZE_KM / 2.0,
                -SMALL_FIELD_SIZE_KM / 2.0,
            ),
            SMALL_FIELD_SIZE_KM,
            SMALL_FIELD_SIZE_KM,
            fill=False,
            edgecolor="black",
            linewidth=2.0,
            zorder=5,
        )
        ax.add_patch(box)

        ax.set_title(f"Large FOV: {variant_names[col]}")
        ax.set_xlabel("x (km)")
        if col == 0:
            ax.set_ylabel("y (km; + downward)")

        cbar = fig.colorbar(
            image,
            ax=ax,
            fraction=0.046,
            pad=0.03,
        )
        cbar.set_label("Amplitude")

    # --------------------------------------------------
    # Row 2: small FOV images + slope arrows
    # --------------------------------------------------
    initial_small_frames = make_three_frames(0, X_SMALL, Y_SMALL)

    small_images = []
    slope_quivers = []
    slope_texts = []

    for col in range(3):
        ax = axes[1, col]

        image = ax.imshow(
            initial_small_frames[col],
            cmap=CMAP,
            vmin=-SMALL_COMMON_VMAX,
            vmax=SMALL_COMMON_VMAX,
            origin="upper",
            extent=[
                -SMALL_FIELD_SIZE_KM / 2.0,
                SMALL_FIELD_SIZE_KM / 2.0,
                SMALL_FIELD_SIZE_KM / 2.0,
                -SMALL_FIELD_SIZE_KM / 2.0,
            ],
            interpolation="nearest",
            aspect="equal",
        )
        small_images.append(image)

        sx0 = plane_slope_x[col, 0]
        sy0 = plane_slope_y[col, 0]
        mag0 = np.hypot(sx0, sy0)

        if mag0 > 0.0:
            arrow_length = MAX_ARROW_LENGTH_KM * mag0 / max_plane_slope
            arrow_u = arrow_length * sx0 / mag0
            arrow_v = arrow_length * sy0 / mag0
        else:
            arrow_u = 0.0
            arrow_v = 0.0

        quiver = ax.quiver(
            [0.0],
            [0.0],
            [arrow_u],
            [arrow_v],
            angles="xy",
            scale_units="xy",
            scale=1.0,
            width=0.010,
            color="black",
            zorder=6,
        )
        slope_quivers.append(quiver)

        slope_text = ax.text(
            0.02,
            0.98,
            "",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9,
            bbox=dict(
                facecolor="white",
                alpha=0.75,
                edgecolor="none",
            ),
            zorder=7,
        )
        slope_texts.append(slope_text)

        ax.set_title(f"500 x 500 km: {variant_names[col]}")
        ax.set_xlabel("x (km)")
        if col == 0:
            ax.set_ylabel("y (km; + downward)")

    small_cbar = fig.colorbar(
        small_images[-1],
        ax=axes[1, :].tolist(),
        fraction=0.018,
        pad=0.015,
    )
    small_cbar.set_label("Amplitude")

    # --------------------------------------------------
    # Row 3: directional keograms
    # --------------------------------------------------
    keogram_images = []
    keogram_time_lines = []

    for col in range(3):
        ax = axes[2, col]

        # imshow expects [distance, time], hence transpose.
        initial_partial = np.full_like(angle_keograms[col].T, np.nan)
        initial_partial[:, :1] = angle_keograms[col, :1].T

        image = ax.imshow(
            initial_partial,
            cmap=CMAP,
            vmin=-SMALL_COMMON_VMAX,
            vmax=SMALL_COMMON_VMAX,
            origin="lower",
            aspect="auto",
            extent=[
                times_min[0],
                times_min[-1],
                distance_axis[0],
                distance_axis[-1],
            ],
            interpolation="nearest",
        )
        keogram_images.append(image)

        current_line = ax.axvline(
            times_min[0],
            color="black",
            linewidth=1.3,
            alpha=0.9,
        )
        keogram_time_lines.append(current_line)

        for requested_distance in PROFILE_DISTANCES_KM:
            ax.axhline(
                requested_distance,
                color="0.35",
                linewidth=0.8,
                linestyle="--",
                alpha=0.7,
            )

        ax.set_title(
            f"{keogram_angle_deg:.0f}° keogram: {variant_names[col]}"
        )
        ax.set_xlim(times_min[0], times_min[-1])
        ax.set_ylim(distance_axis[0], distance_axis[-1])
        ax.set_xlabel("Time (min)")

        if col == 0:
            if np.isclose(keogram_angle_deg % 180.0, 0.0):
                ax.set_ylabel("x distance (km)")
            else:
                ax.set_ylabel("y distance (km; + south)")

    keo_cbar = fig.colorbar(
        keogram_images[-1],
        ax=axes[2, :].tolist(),
        fraction=0.018,
        pad=0.015,
    )
    keo_cbar.set_label("Amplitude")

    # --------------------------------------------------
    # Row 4: profiles at 0, 125, 250 km
    # --------------------------------------------------
    profile_lines = [
        [None] * len(PROFILE_DISTANCES_KM)
        for _ in range(3)
    ]
    profile_time_lines = []

    for col in range(3):
        ax = axes[3, col]

        for profile_index, actual_distance in enumerate(
            angle_profile_distances
        ):
            line, = ax.plot(
                times_min[:1],
                angle_profiles[col, profile_index, :1],
                linewidth=1.5,
                label=f"{actual_distance:.0f} km",
            )
            profile_lines[col][profile_index] = line

        current_line = ax.axvline(
            times_min[0],
            color="black",
            linewidth=1.2,
            alpha=0.8,
        )
        profile_time_lines.append(current_line)

        ax.axhline(
            0.0,
            color="0.5",
            linewidth=0.8,
            linestyle="--",
        )

        ax.set_title(
            f"{keogram_angle_deg:.0f}° profiles: {variant_names[col]}"
        )
        ax.set_xlim(times_min[0], times_min[-1])
        ax.set_ylim(*PROFILE_YLIM)
        ax.set_xlabel("Time (min)")

        if col == 0:
            ax.set_ylabel("Amplitude")

        ax.grid(True, alpha=0.25)
        ax.legend(
            loc="best",
            fontsize=8,
            ncol=3,
        )

    # ==================================================
    # Animation update
    # ==================================================
    def update(frame_index):
        big_frames = make_three_frames(
            frame_index,
            X_BIG,
            Y_BIG,
        )
        small_frames = make_three_frames(
            frame_index,
            X_SMALL,
            Y_SMALL,
        )

        current_time = times_min[frame_index]
        artists = []

        # Row 1
        for col in range(3):
            top_images[col].set_data(big_frames[col])
            artists.append(top_images[col])

        # Row 2: images + plane-fit slope arrows
        for col in range(3):
            small_images[col].set_data(small_frames[col])
            artists.append(small_images[col])

            sx = plane_slope_x[col, frame_index]
            sy = plane_slope_y[col, frame_index]

            magnitude = np.hypot(sx, sy)
            angle = np.degrees(np.arctan2(sy, sx))
            angle = (angle + 180.0) % 360.0 - 180.0

            if magnitude > 0.0:
                arrow_length = (
                    MAX_ARROW_LENGTH_KM
                    * magnitude
                    / max_plane_slope
                )
                arrow_u = arrow_length * sx / magnitude
                arrow_v = arrow_length * sy / magnitude
            else:
                arrow_u = 0.0
                arrow_v = 0.0

            slope_quivers[col].set_UVC(
                [arrow_u],
                [arrow_v],
            )
            artists.append(slope_quivers[col])

            slope_texts[col].set_text(
                f"x slope = {sx:+.4f} /100 km\n"
                f"y slope = {sy:+.4f} /100 km\n"
                f"|slope| = {magnitude:.4f} /100 km\n"
                f"direction = {angle:+.1f}°"
            )
            artists.append(slope_texts[col])

        # Row 3: accumulated keogram through current frame
        for col in range(3):
            partial_keogram = np.full_like(
                angle_keograms[col].T,
                np.nan,
            )
            partial_keogram[:, : frame_index + 1] = (
                angle_keograms[col, : frame_index + 1].T
            )

            keogram_images[col].set_data(partial_keogram)
            keogram_time_lines[col].set_xdata(
                [current_time, current_time]
            )

            artists.extend(
                [
                    keogram_images[col],
                    keogram_time_lines[col],
                ]
            )

        # Row 4: accumulated profiles through current frame
        for col in range(3):
            for profile_index in range(
                len(PROFILE_DISTANCES_KM)
            ):
                profile_lines[col][profile_index].set_data(
                    times_min[: frame_index + 1],
                    angle_profiles[
                        col,
                        profile_index,
                        : frame_index + 1,
                    ],
                )
                artists.append(
                    profile_lines[col][profile_index]
                )

            profile_time_lines[col].set_xdata(
                [current_time, current_time]
            )
            artists.append(profile_time_lines[col])

        bx = bias_x_per_100km[frame_index]
        by = bias_y_per_100km[frame_index]
        wave_angle = propagation_angles_deg[frame_index]

        fig.suptitle(
            "Synthetic wave + changing planar bias\n"
            f"Keogram/profile direction = "
            f"{keogram_angle_deg:.0f}°   |   "
            f"t = {current_time:.0f} min   |   "
            f"wave angle = {wave_angle:.1f}°   |   "
            f"bias = ({bx:+.4f}, {by:+.4f}) amp/100 km",
            fontsize=15,
        )

        return artists

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=NT,
        interval=1000.0 / FPS,
        blit=False,
        repeat=True,
    )

    return fig, ani


# ==================================================
# Save one animation per angle
# ==================================================
def save_animation_for_angle(fig, ani, angle_deg):
    angle_tag = f"{int(round(angle_deg)) % 180:03d}deg"
    preferred = PREFERRED_ANIMATION_FORMAT.lower()

    if (
        preferred == "mp4"
        and animation.writers.is_available("ffmpeg")
    ):
        output_path = (
            OUT_DIR
            / f"{OUTPUT_BASENAME}_keogram_{angle_tag}.mp4"
        )
        print("Saving MP4:", output_path)

        writer = animation.FFMpegWriter(
            fps=FPS,
            bitrate=3500,
            metadata={
                "artist": "synthetic wave/bias animation"
            },
        )
        ani.save(
            output_path,
            writer=writer,
            dpi=ANIMATION_DPI,
        )
    else:
        output_path = (
            OUT_DIR
            / f"{OUTPUT_BASENAME}_keogram_{angle_tag}.gif"
        )
        print(
            "ffmpeg unavailable or GIF requested; "
            "saving GIF:",
            output_path,
        )

        writer = animation.PillowWriter(fps=FPS)
        ani.save(
            output_path,
            writer=writer,
            dpi=ANIMATION_DPI,
        )

    plt.close(fig)
    return output_path


saved_animation_paths = []

for angle_index, keogram_angle_deg in enumerate(
    KEOGRAM_ANGLES_DEG
):
    print(
        f"\nBuilding animation for "
        f"{keogram_angle_deg:.0f}° keogram..."
    )

    fig, ani = build_animation(
        angle_index,
        keogram_angle_deg,
    )

    saved_path = save_animation_for_angle(
        fig,
        ani,
        keogram_angle_deg,
    )

    saved_animation_paths.append(saved_path)


# ==================================================
# Save parameter summary
# ==================================================
parameter_path = OUT_DIR / "animation_parameters.txt"

with open(parameter_path, "w", encoding="utf-8") as file:
    file.write(f"Number of frames: {NT}\n")
    file.write(
        f"Time range (min): "
        f"{times_min[0]} to {times_min[-1]}\n"
    )
    file.write(f"Time step (s): {DT_SECONDS}\n")

    file.write(
        f"Small FOV (km): "
        f"{SMALL_FIELD_SIZE_KM} x {SMALL_FIELD_SIZE_KM}\n"
    )
    file.write(
        f"Small resolution x/y (km): "
        f"{SMALL_DX_KM}, {SMALL_DY_KM}\n"
    )

    file.write(
        f"Big FOV (km): "
        f"{BIG_FIELD_SIZE_KM} x {BIG_FIELD_SIZE_KM}\n"
    )
    file.write(
        f"Big resolution x/y (km): "
        f"{BIG_DX_KM}, {BIG_DY_KM}\n"
    )

    file.write(f"Wavelength (km): {WAVELENGTH_KM}\n")
    file.write(f"Gaussian sigma (km): {GAUSSIAN_WIDTH_KM}\n")
    file.write(f"Maximum wave amplitude: {MAX_AMPLITUDE}\n")
    file.write(f"Wave speed (m/s): {WAVE_SPEED_M_S}\n")
    file.write(f"Wave time shift (min): {WAVE_TIME_SHIFT_MIN}\n")

    file.write(
        "Propagation angle schedule: "
        f"{PROPAGATION_ANGLE_START_DEG} deg at "
        f"{PROPAGATION_ANGLE_START_TIME_MIN} min to "
        f"{PROPAGATION_ANGLE_END_DEG} deg at "
        f"{PROPAGATION_ANGLE_END_TIME_MIN} min\n"
    )

    file.write(
        f"Bias x slope: {BIAS_X_START_PER_100KM} "
        f"at 0 min to {BIAS_X_END_PER_100KM} "
        f"at {BIAS_END_TIME_MIN} min, then held through "
        f"{TOTAL_TIME_MIN} min\n"
    )
    file.write(
        f"Bias y slope: {BIAS_Y_START_PER_100KM} "
        f"at 0 min to {BIAS_Y_END_PER_100KM} "
        f"at {BIAS_END_TIME_MIN} min, then held through "
        f"{TOTAL_TIME_MIN} min\n"
    )

    file.write(
        f"Keogram angles (deg): {KEOGRAM_ANGLES_DEG}\n"
    )
    file.write(
        f"Keogram width (pixels): "
        f"{KEOGRAM_WIDTH_PIXELS}\n"
    )
    file.write(
        f"Requested profile distances (km): "
        f"{PROFILE_DISTANCES_KM}\n"
    )

    for angle_index, angle_deg in enumerate(
        KEOGRAM_ANGLES_DEG
    ):
        file.write(
            f"Actual profile distances for "
            f"{angle_deg:.0f} deg (km): "
            f"{profile_actual_distances[angle_index].tolist()}\n"
        )

    file.write(
        f"Slope vector plot: {slope_plot_path.name}\n"
    )
    file.write(
        f"Calculated bias plot: {calculated_bias_plot_path.name}\n"
    )
    file.write(
        "Calculated bias anchor 1: "
        f"t={CALC_BIAS_FIRST_CENTER_MIN:.1f} min, "
        f"x={calc_bias_anchor_x[0]:+.8f}, "
        f"y={calc_bias_anchor_y[0]:+.8f} amp/100 km\n"
    )
    file.write(
        "Calculated bias anchor 2: "
        f"t={CALC_BIAS_SECOND_CENTER_MIN:.1f} min, "
        f"x={calc_bias_anchor_x[1]:+.8f}, "
        f"y={calc_bias_anchor_y[1]:+.8f} amp/100 km\n"
    )


print("\nFinished.")
for path in saved_animation_paths:
    print("Animation:", path)

print("Slope-vector plot:", slope_plot_path)
print("Calculated-bias plot:", calculated_bias_plot_path)
print(
    "Diagnostics:",
    OUT_DIR / "animation_diagnostics.npz",
)
print("Parameters:", parameter_path)


Precomputing diagnostics for 151 frames...
Calculated-bias anchor at 60 min (average 0-120 min): x=-0.015975, y=+0.004255 amp/100 km
Calculated-bias anchor at 240 min (average 180-300 min): x=+0.025806, y=+0.003711 amp/100 km
Slope-vector plot: synthetic_wave_bias_animation_output/slope_vectors_over_time.png
Calculated-bias plot: synthetic_wave_bias_animation_output/calculated_bias_comparison.png

Building animation for 0° keogram...
Saving MP4: synthetic_wave_bias_animation_output/synthetic_wave_bias_0_to_240min_keogram_000deg.mp4

Building animation for 90° keogram...
Saving MP4: synthetic_wave_bias_animation_output/synthetic_wave_bias_0_to_240min_keogram_090deg.mp4

Finished.
Animation: synthetic_wave_bias_animation_output/synthetic_wave_bias_0_to_240min_keogram_000deg.mp4
Animation: synthetic_wave_bias_animation_output/synthetic_wave_bias_0_to_240min_keogram_090deg.mp4
Slope-vector plot: synthetic_wave_bias_animation_output/slope_vectors_over_time.png
Calculated-bias plot: syntheti

## Synthetic data + Correlation Analysis

In [11]:
# -*- coding: utf-8 -*-
"""
Synthetic wave + time-varying planar bias animation.

For each requested keogram angle (0 and 90 deg), this script saves a separate
4 x 3 animation:

Row 1: Large 5000 x 5000 km FOV, lower spatial resolution
    left   = wave only
    center = bias only
    right  = wave + bias
    A box marks the central 500 x 500 km FOV used below.

Row 2: Central 500 x 500 km FOV at 1 km resolution
    left   = wave only
    center = bias only
    right  = wave + bias
    Each panel includes an arrow showing the plane-fit slope vector
    for the current frame.

Row 3: Current keogram up to the current frame
    left   = wave only
    center = bias only
    right  = wave + bias

Row 4: Time profiles at 0, 125, and 250 km along the current keogram cut
    left   = wave only
    center = bias only
    right  = wave + bias

Two separate static diagnostic figures are also saved:
    1. Plane-fit slope vectors over time. Points show every frame, with very
       faint arrows from the origin; color represents time.
    2. A 4-panel bias comparison showing the fitted wave+bias slopes, the
       calculated bias, the actual imposed bias, and calculated bias - actual bias.

Requested keogram loop:
    0 deg  = E-W cut through y = 0
    90 deg = N-S cut through x = 0

Bias specification:
    at   0 min: x slope = -0.020 amp/100 km, y slope = +0.005 amp/100 km
    at 180 min: x slope = +0.020 amp/100 km, y slope = +0.005 amp/100 km
    x varies linearly from 0 to 180 min and is then held constant through
    the end of the 240-min simulation; y stays constant.

Calculated-bias diagnostic:
    - Average all wave+bias fitted slope points over 0-60 min; assign that
      average to 30 min.
    - Average all wave+bias fitted slope points over 180-240 min; assign that
      average to 210 min.
    - The calculated bias from 30 to 210 min is the straight line connecting
      those two average slope vectors.

Coordinate convention:
    +x = right/east
    +y = down/south
    0 deg = +x/east
    +90 deg = +y/south

At DT_SECONDS = 120 s, including both 0 and 240 min gives 121 frames.
"""

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle

from scipy.ndimage import gaussian_filter, map_coordinates


# ==================================================
# Output
# ==================================================
OUT_DIR = Path("synthetic_wave_bias_animation_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_BASENAME = "synthetic_wave_bias_0_to_240min"

# Animation settings
FPS = 8
ANIMATION_DPI = 110

# "mp4" is preferred. If ffmpeg is unavailable, the script automatically
# falls back to GIF using Pillow.
PREFERRED_ANIMATION_FORMAT = "mp4"


# ==================================================
# Spatial grids
# ==================================================
# Central high-resolution FOV
SMALL_FIELD_SIZE_KM = 500.0
SMALL_DX_KM = 1.0
SMALL_DY_KM = 1.0

# Large overview FOV. 20-km sampling gives 251 x 251 pixels over 5000 km.
BIG_FIELD_SIZE_KM = 5000.0
BIG_DX_KM = 20.0
BIG_DY_KM = 20.0


# ==================================================
# Temporal grid
# ==================================================
DT_SECONDS = 120
TOTAL_TIME_MIN = 300.0

times_s = np.arange(
    0.0,
    TOTAL_TIME_MIN * 60.0 + DT_SECONDS,
    DT_SECONDS,
    dtype=np.float64,
)
times_min = times_s / 60.0
NT = len(times_min)


# ==================================================
# Wave parameters
# ==================================================
WAVELENGTH_KM = 3800.0
GAUSSIAN_WIDTH_KM = 7200.0  # Gaussian sigma
MAX_AMPLITUDE = 0.06

PROPAGATION_ANGLE_START_TIME_MIN = 0.0
PROPAGATION_ANGLE_END_TIME_MIN = 300.0
PROPAGATION_ANGLE_START_DEG = 100.0
PROPAGATION_ANGLE_END_DEG = 100.0

WAVE_SPEED_M_S = 450.0
WAVE_TIME_SHIFT_MIN = 90.0


# ==================================================
# Bias specification
# ==================================================
# Units: perturbation amplitude per 100 km.
BIAS_X_START_PER_100KM = -0.025
BIAS_X_END_PER_100KM = +0.035
BIAS_Y_START_PER_100KM = +0.008
BIAS_Y_END_PER_100KM = +0.004

# Keep the original bias schedule: it reaches its end value at 180 min,
# then remains at that value while the simulation continues to 240 min.
BIAS_END_TIME_MIN = 300.0

# Calculated-bias endpoint windows.
CALC_BIAS_FIRST_CENTER_MIN = 60.0
CALC_BIAS_SECOND_CENTER_MIN = 240.0
CALC_BIAS_HALF_WINDOW_MIN = 60.0


# ==================================================
# Keogram / profile settings
# ==================================================
# Make one complete animation for each of these directions.
KEOGRAM_ANGLES_DEG = [0.0, 90.0]

KEOGRAM_WIDTH_PIXELS = 5
PROFILE_DISTANCES_KM = [0.0, 125.0, 250.0]


# ==================================================
# Plot settings
# ==================================================
CMAP = "RdBu_r"

# Fixed display limits keep colors stable from frame to frame.
TOP_WAVE_VMAX = MAX_AMPLITUDE

# For the large FOV, the largest possible planar bias occurs near a corner.
# abs(bx)*25 + abs(by)*25 because 2500 km = 25 * 100 km.
TOP_BIAS_VMAX = (
    max(abs(BIAS_X_START_PER_100KM), abs(BIAS_X_END_PER_100KM))
    * (BIG_FIELD_SIZE_KM / 2.0 / 100.0)
    + max(abs(BIAS_Y_START_PER_100KM), abs(BIAS_Y_END_PER_100KM))
    * (BIG_FIELD_SIZE_KM / 2.0 / 100.0)
)
TOP_COMBINED_VMAX = TOP_BIAS_VMAX + MAX_AMPLITUDE

# Use one common range for the three 500 x 500 km panels and keograms.
SMALL_COMMON_VMAX = 0.15
PROFILE_YLIM = (-SMALL_COMMON_VMAX, SMALL_COMMON_VMAX)

# Arrow scaling in the image panels. The largest plane-fit slope seen over
# the animation is mapped to this maximum arrow length in km.
MAX_ARROW_LENGTH_KM = 200.0

# Static slope-vector plot.
SLOPE_VECTOR_CMAP = "viridis"
SLOPE_VECTOR_DPI = 250


# ==================================================
# Correlation / phase-speed analysis settings
# ==================================================
# Analyze every 5 degrees from 60 through 160 degrees.
ANALYSIS_ANGLES_DEG = np.arange(60, 161, 5, dtype=int)

# Compare:
#   1. Wave + bias
#   2. Wave + bias - calculated bias
#   3. Wave only
#
# The calculated bias is only defined between its two anchor times, so use
# that same interval for ALL variants to keep the comparison fair.
ANALYSIS_TIME_MIN = CALC_BIAS_FIRST_CENTER_MIN
ANALYSIS_TIME_MAX = CALC_BIAS_SECOND_CENTER_MIN

# Correlate each distance row against the row nearest 0 km.
ANALYSIS_DISTANCE_MIN_KM = 0.0
ANALYSIS_DISTANCE_MAX_KM = 250.0
ANALYSIS_REFERENCE_DISTANCE_KM = 0.0

# Shift range used for the correlation map and ridge fit.
ANALYSIS_SHIFT_MIN_MIN = -50.0
ANALYSIS_SHIFT_MAX_MIN = +50.0

# The 0-km autocorrelation profile uses a wider range, matching the analysis
# style of the separate correlation script.
ANALYSIS_ZERO_KM_SHIFT_FACTOR = 4.0
ANALYSIS_ZERO_KM_SHIFT_MIN_MIN = (
    ANALYSIS_ZERO_KM_SHIFT_FACTOR * ANALYSIS_SHIFT_MIN_MIN
)
ANALYSIS_ZERO_KM_SHIFT_MAX_MIN = (
    ANALYSIS_ZERO_KM_SHIFT_FACTOR * ANALYSIS_SHIFT_MAX_MIN
)

ANALYSIS_MIN_VALID_POINTS = 20
ANALYSIS_MIN_CORRELATION_FOR_SLOPE = 0.50

# Optional NaN-aware 2-D smoothing before the correlation calculation.
ANALYSIS_APPLY_2D_SMOOTHING = True
ANALYSIS_SMOOTH_SIGMA_TIME_PIXELS = 1.5
ANALYSIS_SMOOTH_SIGMA_DISTANCE_PIXELS = 2.0

# Ridge cleanup / fit settings.
ANALYSIS_REMOVE_RIDGE_OUTLIERS = True
ANALYSIS_RIDGE_MAXIMUM_JUMP_MIN = 20.0
ANALYSIS_RIDGE_OUTLIER_SIGMA = 3.0
ANALYSIS_RIDGE_OUTLIER_MAX_ITERATIONS = 10
ANALYSIS_RIDGE_OUTLIER_MIN_POINTS = 3

ANALYSIS_DPI = 200
ANALYSIS_CORRELATION_CMAP = "RdBu_r"
ANALYSIS_CORRELATION_VMIN = 0.70
ANALYSIS_CORRELATION_VMAX = 1.00

ANALYSIS_OUT_DIR = OUT_DIR / "correlation_analysis_60_to_160deg"
ANALYSIS_OUT_DIR.mkdir(parents=True, exist_ok=True)


# ==================================================
# Coordinate grids
# ==================================================
def centered_grid(field_size_km, dx_km, dy_km):
    nx = int(round(field_size_km / dx_km)) + 1
    ny = int(round(field_size_km / dy_km)) + 1

    x = np.linspace(-field_size_km / 2.0, field_size_km / 2.0, nx)
    y = np.linspace(-field_size_km / 2.0, field_size_km / 2.0, ny)
    X, Y = np.meshgrid(x, y)
    return x, y, X, Y


x_small, y_small, X_SMALL, Y_SMALL = centered_grid(
    SMALL_FIELD_SIZE_KM,
    SMALL_DX_KM,
    SMALL_DY_KM,
)

x_big, y_big, X_BIG, Y_BIG = centered_grid(
    BIG_FIELD_SIZE_KM,
    BIG_DX_KM,
    BIG_DY_KM,
)

NY_SMALL, NX_SMALL = X_SMALL.shape


# ==================================================
# Time-varying wave state
# ==================================================
def propagation_angle_deg_at_time(time_min):
    """Prescribed propagation angle."""
    t = np.asarray(time_min, dtype=np.float64)

    fraction = (
        t - PROPAGATION_ANGLE_START_TIME_MIN
    ) / (
        PROPAGATION_ANGLE_END_TIME_MIN
        - PROPAGATION_ANGLE_START_TIME_MIN
    )
    fraction = np.clip(fraction, 0.0, 1.0)

    return (
        PROPAGATION_ANGLE_START_DEG
        + fraction
        * (PROPAGATION_ANGLE_END_DEG - PROPAGATION_ANGLE_START_DEG)
    )


propagation_angles_deg = propagation_angle_deg_at_time(times_min)
propagation_angles_rad = np.deg2rad(propagation_angles_deg)

wavenumber_rad_per_km = 2.0 * np.pi / WAVELENGTH_KM
wave_speed_km_s = WAVE_SPEED_M_S / 1000.0

# Integrate the moving packet-center trajectory.
velocity_x_km_s = wave_speed_km_s * np.cos(propagation_angles_rad)
velocity_y_km_s = wave_speed_km_s * np.sin(propagation_angles_rad)

packet_center_x_km = np.zeros(NT, dtype=np.float64)
packet_center_y_km = np.zeros(NT, dtype=np.float64)

if NT > 1:
    dt_integration_s = np.diff(times_s)
    packet_center_x_km[1:] = np.cumsum(
        0.5
        * (velocity_x_km_s[:-1] + velocity_x_km_s[1:])
        * dt_integration_s
    )
    packet_center_y_km[1:] = np.cumsum(
        0.5
        * (velocity_y_km_s[:-1] + velocity_y_km_s[1:])
        * dt_integration_s
    )

# Carrier-wave temporal phase.
angular_frequency_rad_s = wavenumber_rad_per_km * wave_speed_km_s

temporal_phase_rad = angular_frequency_rad_s * times_s
phase_shift_rad = (
    angular_frequency_rad_s
    * WAVE_TIME_SHIFT_MIN
    * 60.0
)
temporal_phase_rad = temporal_phase_rad + phase_shift_rad


# ==================================================
# Time-varying bias state
# ==================================================
bias_fraction = np.clip(
    times_min / BIAS_END_TIME_MIN,
    0.0,
    1.0,
)

bias_x_per_100km = (
    BIAS_X_START_PER_100KM
    + bias_fraction
    * (BIAS_X_END_PER_100KM - BIAS_X_START_PER_100KM)
)

bias_y_per_100km = (
    BIAS_Y_START_PER_100KM
    + bias_fraction
    * (BIAS_Y_END_PER_100KM - BIAS_Y_START_PER_100KM)
)

bias_magnitude_per_100km = np.hypot(
    bias_x_per_100km,
    bias_y_per_100km,
)

bias_angle_deg = np.degrees(
    np.arctan2(bias_y_per_100km, bias_x_per_100km)
)
bias_angle_deg = (bias_angle_deg + 180.0) % 360.0 - 180.0


# ==================================================
# Frame generators
# ==================================================
def make_wave_frame(frame_index, X, Y):
    """Generate one wave frame on an arbitrary centered X/Y grid."""
    theta = propagation_angles_rad[frame_index]
    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)

    relative_x_km = X - packet_center_x_km[frame_index]
    relative_y_km = Y - packet_center_y_km[frame_index]

    parallel_relative_km = (
        relative_x_km * cos_theta
        + relative_y_km * sin_theta
    )

    gaussian_envelope = np.exp(
        -0.5 * (parallel_relative_km / GAUSSIAN_WIDTH_KM) ** 2
    )

    spatial_phase_rad = wavenumber_rad_per_km * (
        X * cos_theta + Y * sin_theta
    )

    wave_phase_rad = spatial_phase_rad - temporal_phase_rad[frame_index]

    return (
        MAX_AMPLITUDE
        * gaussian_envelope
        * np.cos(wave_phase_rad)
    ).astype(np.float32)


def make_bias_frame(frame_index, X, Y):
    """Generate one planar-bias frame on an arbitrary centered X/Y grid."""
    bx = bias_x_per_100km[frame_index]
    by = bias_y_per_100km[frame_index]

    return (
        bx * X / 100.0
        + by * Y / 100.0
    ).astype(np.float32)


def make_three_frames(frame_index, X, Y):
    """Return wave, bias, and wave+bias for one time/frame."""
    wave = make_wave_frame(frame_index, X, Y)
    bias = make_bias_frame(frame_index, X, Y)
    combined = wave + bias
    return wave, bias, combined


# ==================================================
# Plane-fit slope helpers
# ==================================================
# For a centered rectangular grid, x and y are orthogonal and zero-mean.
X_SMALL_64 = X_SMALL.astype(np.float64)
Y_SMALL_64 = Y_SMALL.astype(np.float64)
SUM_X2 = np.sum(X_SMALL_64 ** 2)
SUM_Y2 = np.sum(Y_SMALL_64 ** 2)


def fit_plane_slope(frame):
    """
    Fit z = intercept + slope_x*x + slope_y*y.

    Returns slope_x and slope_y in amplitude per 100 km.
    """
    z = np.asarray(frame, dtype=np.float64)

    slope_x_per_km = np.sum(X_SMALL_64 * z) / SUM_X2
    slope_y_per_km = np.sum(Y_SMALL_64 * z) / SUM_Y2

    return (
        100.0 * slope_x_per_km,
        100.0 * slope_y_per_km,
    )


# ==================================================
# Directional keogram helper
# ==================================================
def extract_directional_cut(frame, angle_deg, width_pixels=7):
    """
    Extract a center cut at 0 or 90 degrees.

    0 deg:
        E-W cut through y = 0.
        Average a horizontal strip across y.
        Returned distance coordinate is x.

    90 deg:
        N-S cut through x = 0.
        Average a vertical strip across x.
        Returned distance coordinate is y.

    Returns
    -------
    distance_km : 1-D ndarray
    cut : 1-D ndarray
    """
    angle = float(angle_deg) % 180.0
    half_width = width_pixels // 2

    if np.isclose(angle, 0.0):
        center_row = (frame.shape[0] - 1) // 2
        row_start = max(0, center_row - half_width)
        row_stop = min(frame.shape[0], center_row + half_width + 1)

        cut = np.nanmean(frame[row_start:row_stop, :], axis=0)
        return x_small, cut

    if np.isclose(angle, 90.0):
        center_col = (frame.shape[1] - 1) // 2
        col_start = max(0, center_col - half_width)
        col_stop = min(frame.shape[1], center_col + half_width + 1)

        cut = np.nanmean(frame[:, col_start:col_stop], axis=1)
        return y_small, cut

    raise ValueError(
        "This script currently supports only 0 and 90 degree cuts. "
        f"Received {angle_deg} deg."
    )


# ==================================================
# Precompute small-FOV diagnostics
# ==================================================
variant_names = ["Wave only", "Bias only", "Wave + bias"]

N_ANGLES = len(KEOGRAM_ANGLES_DEG)

# The 0 and 90 deg cuts have the same number of samples here because
# SMALL_DX_KM == SMALL_DY_KM and the FOV is square.
N_CUT = NX_SMALL

keograms = np.empty(
    (N_ANGLES, 3, NT, N_CUT),
    dtype=np.float32,
)

plane_slope_x = np.empty((3, NT), dtype=np.float64)
plane_slope_y = np.empty((3, NT), dtype=np.float64)

print(f"Precomputing diagnostics for {NT} frames...")

for it in range(NT):
    small_frames = make_three_frames(it, X_SMALL, Y_SMALL)

    for variant_index, frame in enumerate(small_frames):
        sx, sy = fit_plane_slope(frame)
        plane_slope_x[variant_index, it] = sx
        plane_slope_y[variant_index, it] = sy

        for angle_index, angle_deg in enumerate(KEOGRAM_ANGLES_DEG):
            _, cut = extract_directional_cut(
                frame,
                angle_deg=angle_deg,
                width_pixels=KEOGRAM_WIDTH_PIXELS,
            )
            keograms[angle_index, variant_index, it] = cut


plane_slope_magnitude = np.hypot(plane_slope_x, plane_slope_y)
max_plane_slope = np.nanmax(plane_slope_magnitude)

if not np.isfinite(max_plane_slope) or max_plane_slope <= 0.0:
    max_plane_slope = 1.0


# ==================================================
# Calculate bias from the fitted wave+bias slope
# ==================================================
# Variant index 2 is "Wave + bias".
combined_slope_x = plane_slope_x[2]
combined_slope_y = plane_slope_y[2]

first_window_start_min = (
    CALC_BIAS_FIRST_CENTER_MIN - CALC_BIAS_HALF_WINDOW_MIN
)
first_window_end_min = (
    CALC_BIAS_FIRST_CENTER_MIN + CALC_BIAS_HALF_WINDOW_MIN
)
second_window_start_min = (
    CALC_BIAS_SECOND_CENTER_MIN - CALC_BIAS_HALF_WINDOW_MIN
)
second_window_end_min = (
    CALC_BIAS_SECOND_CENTER_MIN + CALC_BIAS_HALF_WINDOW_MIN
)

first_window_mask = (
    (times_min >= first_window_start_min)
    & (times_min <= first_window_end_min)
)
second_window_mask = (
    (times_min >= second_window_start_min)
    & (times_min <= second_window_end_min)
)

calc_bias_anchor_x = np.array(
    [
        np.nanmean(combined_slope_x[first_window_mask]),
        np.nanmean(combined_slope_x[second_window_mask]),
    ],
    dtype=np.float64,
)
calc_bias_anchor_y = np.array(
    [
        np.nanmean(combined_slope_y[first_window_mask]),
        np.nanmean(combined_slope_y[second_window_mask]),
    ],
    dtype=np.float64,
)
calc_bias_anchor_times_min = np.array(
    [
        CALC_BIAS_FIRST_CENTER_MIN,
        CALC_BIAS_SECOND_CENTER_MIN,
    ],
    dtype=np.float64,
)

# Only define the calculated bias on the requested 30-210 min interval.
calc_bias_x_per_100km = np.full(NT, np.nan, dtype=np.float64)
calc_bias_y_per_100km = np.full(NT, np.nan, dtype=np.float64)

calc_bias_interval_mask = (
    (times_min >= CALC_BIAS_FIRST_CENTER_MIN)
    & (times_min <= CALC_BIAS_SECOND_CENTER_MIN)
)

calc_fraction = (
    times_min[calc_bias_interval_mask] - CALC_BIAS_FIRST_CENTER_MIN
) / (
    CALC_BIAS_SECOND_CENTER_MIN - CALC_BIAS_FIRST_CENTER_MIN
)

calc_bias_x_per_100km[calc_bias_interval_mask] = (
    calc_bias_anchor_x[0]
    + calc_fraction
    * (calc_bias_anchor_x[1] - calc_bias_anchor_x[0])
)
calc_bias_y_per_100km[calc_bias_interval_mask] = (
    calc_bias_anchor_y[0]
    + calc_fraction
    * (calc_bias_anchor_y[1] - calc_bias_anchor_y[0])
)

# Difference between the calculated and actual imposed bias.
# These remain NaN outside the interval where calculated bias is defined.
calc_bias_minus_actual_x_per_100km = (
    calc_bias_x_per_100km - bias_x_per_100km
)
calc_bias_minus_actual_y_per_100km = (
    calc_bias_y_per_100km - bias_y_per_100km
)

print(
    "Calculated-bias anchor at "
    f"{CALC_BIAS_FIRST_CENTER_MIN:.0f} min "
    f"(average {first_window_start_min:.0f}-"
    f"{first_window_end_min:.0f} min): "
    f"x={calc_bias_anchor_x[0]:+.6f}, "
    f"y={calc_bias_anchor_y[0]:+.6f} amp/100 km"
)
print(
    "Calculated-bias anchor at "
    f"{CALC_BIAS_SECOND_CENTER_MIN:.0f} min "
    f"(average {second_window_start_min:.0f}-"
    f"{second_window_end_min:.0f} min): "
    f"x={calc_bias_anchor_x[1]:+.6f}, "
    f"y={calc_bias_anchor_y[1]:+.6f} amp/100 km"
)


# ==================================================
# Profiles for each cut angle
# ==================================================
# distances_by_angle has shape [N_ANGLES, N_CUT].
distances_by_angle = np.empty((N_ANGLES, N_CUT), dtype=np.float64)

# profiles shape:
# [angle, variant, requested profile distance, time]
profiles = np.empty(
    (
        N_ANGLES,
        3,
        len(PROFILE_DISTANCES_KM),
        NT,
    ),
    dtype=np.float32,
)

profile_actual_distances = np.empty(
    (N_ANGLES, len(PROFILE_DISTANCES_KM)),
    dtype=np.float64,
)

for angle_index, angle_deg in enumerate(KEOGRAM_ANGLES_DEG):
    if np.isclose(angle_deg % 180.0, 0.0):
        distance_axis = x_small
    elif np.isclose(angle_deg % 180.0, 90.0):
        distance_axis = y_small
    else:
        raise ValueError("Only 0 and 90 degree cuts are supported.")

    distances_by_angle[angle_index] = distance_axis

    for profile_index, requested_distance in enumerate(PROFILE_DISTANCES_KM):
        distance_index = int(
            np.argmin(np.abs(distance_axis - requested_distance))
        )
        profile_actual_distances[angle_index, profile_index] = (
            distance_axis[distance_index]
        )

        for variant_index in range(3):
            profiles[
                angle_index,
                variant_index,
                profile_index,
            ] = keograms[
                angle_index,
                variant_index,
                :,
                distance_index,
            ]


# ==================================================
# Save useful diagnostic arrays
# ==================================================
np.savez_compressed(
    OUT_DIR / "animation_diagnostics.npz",
    times_min=times_min,
    x_small_km=x_small,
    y_small_km=y_small,
    keogram_angles_deg=np.asarray(KEOGRAM_ANGLES_DEG, dtype=float),
    keogram_distances_km=distances_by_angle,
    propagation_angles_deg=propagation_angles_deg,
    packet_center_x_km=packet_center_x_km,
    packet_center_y_km=packet_center_y_km,
    bias_x_per_100km=bias_x_per_100km,
    bias_y_per_100km=bias_y_per_100km,
    bias_angle_deg=bias_angle_deg,
    bias_magnitude_per_100km=bias_magnitude_per_100km,
    plane_slope_x_per_100km=plane_slope_x,
    plane_slope_y_per_100km=plane_slope_y,
    calculated_bias_anchor_times_min=calc_bias_anchor_times_min,
    calculated_bias_anchor_x_per_100km=calc_bias_anchor_x,
    calculated_bias_anchor_y_per_100km=calc_bias_anchor_y,
    calculated_bias_x_per_100km=calc_bias_x_per_100km,
    calculated_bias_y_per_100km=calc_bias_y_per_100km,
    calculated_bias_minus_actual_x_per_100km=(
        calc_bias_minus_actual_x_per_100km
    ),
    calculated_bias_minus_actual_y_per_100km=(
        calc_bias_minus_actual_y_per_100km
    ),
    keograms=keograms,
    profile_requested_distances_km=np.asarray(PROFILE_DISTANCES_KM),
    profile_actual_distances_km=profile_actual_distances,
    profiles=profiles,
)


# ==================================================
# Separate slope-arrow-over-time plot
# ==================================================
def save_slope_vector_plot():
    """
    Plot every frame's plane-fit slope tip as a point.

    Very faint arrows from the origin are retained only to make the vector
    direction easy to see. Point/arrow color represents time.
    """
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18.5, 5.3),
        sharex=True,
        sharey=True,
    )

    norm = Normalize(vmin=times_min[0], vmax=times_min[-1])
    cmap = plt.get_cmap(SLOPE_VECTOR_CMAP)

    component_limit = np.nanmax(
        np.abs(
            np.concatenate(
                [
                    plane_slope_x.ravel(),
                    plane_slope_y.ravel(),
                ]
            )
        )
    )

    if not np.isfinite(component_limit) or component_limit <= 0.0:
        component_limit = 1.0

    component_limit *= 1.10

    point_size = 20

    for col, ax in enumerate(axes):
        # Very faint arrows so the dense plot is not dominated by arrowheads.
        ax.quiver(
            np.zeros(NT),
            np.zeros(NT),
            plane_slope_x[col],
            plane_slope_y[col],
            times_min,
            cmap=cmap,
            norm=norm,
            angles="xy",
            scale_units="xy",
            scale=1.0,
            width=0.0020,
            headwidth=3.0,
            headlength=4.0,
            headaxislength=3.5,
            alpha=0.12,
            zorder=1,
        )

        # Main representation: slope-vector tips as time-colored points.
        ax.scatter(
            plane_slope_x[col],
            plane_slope_y[col],
            c=times_min,
            cmap=cmap,
            norm=norm,
            s=point_size,
            linewidths=0.0,
            alpha=0.92,
            zorder=3,
        )

        # Thin trajectory through the slope-vector tips.
        ax.plot(
            plane_slope_x[col],
            plane_slope_y[col],
            linewidth=0.75,
            alpha=0.28,
            color="0.25",
            zorder=2,
        )

        # Mark the first and last points.
        ax.scatter(
            [plane_slope_x[col, 0]],
            [plane_slope_y[col, 0]],
            s=46,
            facecolors="none",
            edgecolors="black",
            linewidths=1.1,
            zorder=5,
            label="0 min",
        )
        ax.scatter(
            [plane_slope_x[col, -1]],
            [plane_slope_y[col, -1]],
            s=44,
            marker="x",
            color="black",
            linewidths=1.2,
            zorder=5,
            label=f"{times_min[-1]:.0f} min",
        )

        ax.axhline(0.0, color="0.6", linewidth=0.8)
        ax.axvline(0.0, color="0.6", linewidth=0.8)

        ax.set_xlim(-component_limit, component_limit)

        # +y is downward in the image coordinate convention.
        ax.set_ylim(component_limit, -component_limit)

        ax.set_aspect("equal", adjustable="box")
        ax.grid(True, alpha=0.22)
        ax.set_title(variant_names[col])
        ax.set_xlabel("x slope (amp/100 km)")

        if col == 0:
            ax.set_ylabel("y slope (amp/100 km; + downward)")

        ax.legend(loc="best", fontsize=8)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    # Reserve a dedicated strip on the right so the colorbar cannot overlap
    # the third subplot.
    cbar_ax = fig.add_axes([0.935, 0.18, 0.012, 0.66])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label("Time (min)")

    fig.suptitle(
        "Plane-fit slope vectors over time",
        fontsize=14,
    )

    fig.subplots_adjust(
        top=0.86,
        bottom=0.14,
        left=0.06,
        right=0.90,
        wspace=0.20,
    )

    output_path = OUT_DIR / "slope_vectors_over_time.png"
    fig.savefig(
        output_path,
        dpi=SLOPE_VECTOR_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

    print("Slope-vector plot:", output_path)
    return output_path


# ==================================================
# Calculated-bias comparison plot
# ==================================================
def save_calculated_bias_plot():
    """
    Save a 4-panel slope-vector-space comparison:

    1. Plane-fit slope of wave + actual bias.
    2. Calculated bias: straight line between the two averaged anchors.
    3. Actual imposed bias for comparison.
    4. Difference: calculated bias - actual imposed bias.
    """
    fig, axes = plt.subplots(
        1,
        4,
        figsize=(24.0, 5.3),
        sharex=True,
        sharey=True,
    )

    norm = Normalize(vmin=times_min[0], vmax=times_min[-1])
    cmap = plt.get_cmap(SLOPE_VECTOR_CMAP)

    finite_calc = (
        np.isfinite(calc_bias_x_per_100km)
        & np.isfinite(calc_bias_y_per_100km)
    )
    finite_difference = (
        np.isfinite(calc_bias_minus_actual_x_per_100km)
        & np.isfinite(calc_bias_minus_actual_y_per_100km)
    )

    all_x = np.concatenate(
        [
            combined_slope_x,
            calc_bias_x_per_100km[finite_calc],
            bias_x_per_100km,
            calc_bias_minus_actual_x_per_100km[finite_difference],
        ]
    )
    all_y = np.concatenate(
        [
            combined_slope_y,
            calc_bias_y_per_100km[finite_calc],
            bias_y_per_100km,
            calc_bias_minus_actual_y_per_100km[finite_difference],
        ]
    )

    component_limit = np.nanmax(np.abs(np.concatenate([all_x, all_y])))
    if not np.isfinite(component_limit) or component_limit <= 0.0:
        component_limit = 1.0
    component_limit *= 1.12

    def draw_vector_points(
        ax,
        x_values,
        y_values,
        color_times,
        point_alpha=0.92,
        arrow_alpha=0.10,
    ):
        ax.quiver(
            np.zeros_like(x_values),
            np.zeros_like(y_values),
            x_values,
            y_values,
            color_times,
            cmap=cmap,
            norm=norm,
            angles="xy",
            scale_units="xy",
            scale=1.0,
            width=0.0020,
            headwidth=3.0,
            headlength=4.0,
            headaxislength=3.5,
            alpha=arrow_alpha,
            zorder=1,
        )
        ax.scatter(
            x_values,
            y_values,
            c=color_times,
            cmap=cmap,
            norm=norm,
            s=22,
            linewidths=0.0,
            alpha=point_alpha,
            zorder=3,
        )
        ax.plot(
            x_values,
            y_values,
            color="0.25",
            linewidth=0.8,
            alpha=0.30,
            zorder=2,
        )

    # Panel 1: measured/fitted slope of wave + bias.
    draw_vector_points(
        axes[0],
        combined_slope_x,
        combined_slope_y,
        times_min,
        point_alpha=0.92,
        arrow_alpha=0.10,
    )
    axes[0].set_title("Wave + bias plane-fit slope")

    # Show which points enter the two endpoint averages.
    axes[0].scatter(
        combined_slope_x[first_window_mask],
        combined_slope_y[first_window_mask],
        s=52,
        facecolors="none",
        edgecolors="black",
        linewidths=0.7,
        alpha=0.45,
        zorder=4,
        label="0-120 and 180-300 min windows",
    )
    axes[0].scatter(
        combined_slope_x[second_window_mask],
        combined_slope_y[second_window_mask],
        s=52,
        facecolors="none",
        edgecolors="black",
        linewidths=0.7,
        alpha=0.45,
        zorder=4,
    )
    axes[0].legend(loc="best", fontsize=8)

    # Panel 2: calculated bias.
    calc_times = times_min[finite_calc]
    calc_x = calc_bias_x_per_100km[finite_calc]
    calc_y = calc_bias_y_per_100km[finite_calc]

    draw_vector_points(
        axes[1],
        calc_x,
        calc_y,
        calc_times,
        point_alpha=0.95,
        arrow_alpha=0.10,
    )
    axes[1].scatter(
        calc_bias_anchor_x,
        calc_bias_anchor_y,
        c=calc_bias_anchor_times_min,
        cmap=cmap,
        norm=norm,
        s=90,
        edgecolors="black",
        linewidths=1.0,
        zorder=5,
    )
    axes[1].set_title(
        f"Calculated bias "
        f"({CALC_BIAS_FIRST_CENTER_MIN:.0f}-"
        f"{CALC_BIAS_SECOND_CENTER_MIN:.0f} min)"
    )
    axes[1].text(
        0.03,
        0.97,
        f"{CALC_BIAS_FIRST_CENTER_MIN:.0f} min = mean of "
        f"{first_window_start_min:.0f}-{first_window_end_min:.0f} min\n"
        f"{CALC_BIAS_SECOND_CENTER_MIN:.0f} min = mean of "
        f"{second_window_start_min:.0f}-{second_window_end_min:.0f} min",
        transform=axes[1].transAxes,
        ha="left",
        va="top",
        fontsize=8.5,
        bbox=dict(
            facecolor="white",
            alpha=0.78,
            edgecolor="none",
        ),
        zorder=6,
    )

    # Panel 3: actual imposed bias.
    draw_vector_points(
        axes[2],
        bias_x_per_100km,
        bias_y_per_100km,
        times_min,
        point_alpha=0.92,
        arrow_alpha=0.10,
    )
    axes[2].set_title("Actual bias")

    # Panel 4: calculated bias - actual bias.
    difference_times = times_min[finite_difference]
    difference_x = calc_bias_minus_actual_x_per_100km[finite_difference]
    difference_y = calc_bias_minus_actual_y_per_100km[finite_difference]

    draw_vector_points(
        axes[3],
        difference_x,
        difference_y,
        difference_times,
        point_alpha=0.95,
        arrow_alpha=0.10,
    )
    axes[3].scatter(
        [0.0],
        [0.0],
        s=75,
        facecolors="none",
        edgecolors="black",
        linewidths=1.2,
        zorder=5,
        label="Perfect agreement",
    )
    axes[3].set_title("Calculated bias - actual bias")
    axes[3].legend(loc="best", fontsize=8)

    for col, ax in enumerate(axes):
        ax.axhline(0.0, color="0.6", linewidth=0.8)
        ax.axvline(0.0, color="0.6", linewidth=0.8)
        ax.set_xlim(-component_limit, component_limit)
        ax.set_ylim(component_limit, -component_limit)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(True, alpha=0.22)
        ax.set_xlabel("x slope (amp/100 km)")

        if col == 0:
            ax.set_ylabel("y slope (amp/100 km; + downward)")

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar_ax = fig.add_axes([0.947, 0.18, 0.010, 0.66])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label("Time (min)")

    fig.suptitle(
        "Calculated bias from wave+bias plane-fit slopes",
        fontsize=14,
    )

    fig.subplots_adjust(
        top=0.86,
        bottom=0.14,
        left=0.045,
        right=0.925,
        wspace=0.20,
    )

    output_path = OUT_DIR / "calculated_bias_comparison.png"
    fig.savefig(
        output_path,
        dpi=SLOPE_VECTOR_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

    print("Calculated-bias plot:", output_path)
    return output_path


slope_plot_path = save_slope_vector_plot()
calculated_bias_plot_path = save_calculated_bias_plot()


# ==================================================
# Correlation / phase-speed analysis helpers
# ==================================================
def extract_directional_cut_any_angle(
    frame,
    angle_deg,
    width_pixels=7,
):
    """
    Extract a center directional cut for any angle.

    Coordinate convention:
        0 deg   = +x/east
        +90 deg = +y/south

    The cut is sampled every 1 km from -250 to +250 km. A strip with the
    requested width is averaged perpendicular to the cut. Bilinear
    interpolation is used so angles do not need to fall on grid axes.
    """
    frame = np.asarray(frame, dtype=float)
    theta = np.deg2rad(float(angle_deg))

    distance_km = np.arange(
        -SMALL_FIELD_SIZE_KM / 2.0,
        SMALL_FIELD_SIZE_KM / 2.0 + SMALL_DX_KM,
        SMALL_DX_KM,
        dtype=float,
    )

    half_width = int(width_pixels) // 2
    width_offsets_km = (
        np.arange(-half_width, half_width + 1, dtype=float)
        * SMALL_DX_KM
    )

    # Unit vectors parallel and perpendicular to the requested slice.
    ux = np.cos(theta)
    uy = np.sin(theta)
    vx = -np.sin(theta)
    vy = np.cos(theta)

    x_parallel = distance_km * ux
    y_parallel = distance_km * uy

    sampled_strips = []

    for offset_km in width_offsets_km:
        sample_x = x_parallel + offset_km * vx
        sample_y = y_parallel + offset_km * vy

        sample_col = (
            (sample_x - x_small[0]) / SMALL_DX_KM
        )
        sample_row = (
            (sample_y - y_small[0]) / SMALL_DY_KM
        )

        sampled = map_coordinates(
            frame,
            [sample_row, sample_col],
            order=1,
            mode="constant",
            cval=np.nan,
        )
        sampled_strips.append(sampled)

    sampled_strips = np.asarray(sampled_strips, dtype=float)

    with np.errstate(invalid="ignore"):
        cut = np.nanmean(sampled_strips, axis=0)

    return distance_km, cut


def make_calculated_bias_frame(frame_index, X, Y):
    """
    Construct the plane represented by the calculated bias slope.

    Outside the calculated-bias interval the stored slope is NaN, so this
    function also returns an all-NaN plane there.
    """
    bx = calc_bias_x_per_100km[frame_index]
    by = calc_bias_y_per_100km[frame_index]

    if not (np.isfinite(bx) and np.isfinite(by)):
        return np.full(X.shape, np.nan, dtype=np.float32)

    return (
        bx * X / 100.0
        + by * Y / 100.0
    ).astype(np.float32)


def smooth_keogram_2d_nanaware(
    data,
    sigma_time,
    sigma_distance,
):
    """Apply a NaN-aware 2-D Gaussian filter."""
    data = np.asarray(data, dtype=float)
    valid = np.isfinite(data)

    filled = np.where(valid, data, 0.0)
    weights = valid.astype(float)

    sigma = (float(sigma_time), float(sigma_distance))

    filtered_values = gaussian_filter(
        filled,
        sigma=sigma,
        mode="nearest",
    )
    filtered_weights = gaussian_filter(
        weights,
        sigma=sigma,
        mode="nearest",
    )

    output = np.full_like(data, np.nan, dtype=float)
    good = filtered_weights > 1e-12
    output[good] = (
        filtered_values[good]
        / filtered_weights[good]
    )
    return output


def nan_pearson_correlation(
    reference,
    comparison,
    minimum_points=20,
):
    """Pearson correlation using only finite overlapping values."""
    reference = np.asarray(reference, dtype=float)
    comparison = np.asarray(comparison, dtype=float)

    valid = np.isfinite(reference) & np.isfinite(comparison)

    if np.count_nonzero(valid) < minimum_points:
        return np.nan

    x = reference[valid]
    y = comparison[valid]

    x = x - np.mean(x)
    y = y - np.mean(y)

    denominator = np.sqrt(
        np.sum(x ** 2) * np.sum(y ** 2)
    )

    if not np.isfinite(denominator) or denominator <= 0.0:
        return np.nan

    return float(np.clip(
        np.sum(x * y) / denominator,
        -1.0,
        1.0,
    ))


def calculate_shifted_correlation(
    reference_row,
    comparison_row,
    shift_pixels,
    minimum_points=20,
):
    """
    Shift comparison_row and correlate it with reference_row.

    Positive shift means the comparison row is moved toward later times.
    Circular wrapping is not used.
    """
    reference_row = np.asarray(reference_row, dtype=float)
    comparison_row = np.asarray(comparison_row, dtype=float)

    if reference_row.shape != comparison_row.shape:
        raise ValueError(
            "Reference and comparison rows must have the same shape."
        )

    ntime = len(reference_row)

    if abs(int(shift_pixels)) >= ntime:
        return np.nan

    if shift_pixels > 0:
        reference_overlap = reference_row[shift_pixels:]
        comparison_overlap = comparison_row[:-shift_pixels]
    elif shift_pixels < 0:
        k = abs(int(shift_pixels))
        reference_overlap = reference_row[:-k]
        comparison_overlap = comparison_row[k:]
    else:
        reference_overlap = reference_row
        comparison_overlap = comparison_row

    return nan_pearson_correlation(
        reference_overlap,
        comparison_overlap,
        minimum_points=minimum_points,
    )


def calculate_correlation_map(
    keogram,
    distance_km,
    reference_distance_km,
    selected_distance_mask,
    shifts_pixels,
    minimum_points=20,
):
    """
    Correlate every selected distance row with the reference row.

    This version evaluates all selected distance rows at once for each
    time shift. It gives the same Pearson-correlation result as looping
    row by row, but is much faster for the 29-angle analysis.
    """
    keogram = np.asarray(keogram, dtype=float)
    distance_km = np.asarray(distance_km, dtype=float)

    reference_index = int(
        np.argmin(np.abs(distance_km - reference_distance_km))
    )
    reference_row = keogram[:, reference_index]

    selected_indices = np.flatnonzero(selected_distance_mask)
    selected_distances = distance_km[selected_indices]
    selected_data = keogram[:, selected_indices]

    correlation_map = np.full(
        (len(selected_indices), len(shifts_pixels)),
        np.nan,
        dtype=float,
    )

    for shift_i, shift_pixels in enumerate(shifts_pixels):
        shift_pixels = int(shift_pixels)

        if abs(shift_pixels) >= keogram.shape[0]:
            continue

        if shift_pixels > 0:
            reference_overlap = reference_row[shift_pixels:]
            comparison_overlap = selected_data[:-shift_pixels, :]
        elif shift_pixels < 0:
            k = abs(shift_pixels)
            reference_overlap = reference_row[:-k]
            comparison_overlap = selected_data[k:, :]
        else:
            reference_overlap = reference_row
            comparison_overlap = selected_data

        reference_2d = reference_overlap[:, np.newaxis]

        valid = (
            np.isfinite(reference_2d)
            & np.isfinite(comparison_overlap)
        )

        counts = np.sum(valid, axis=0)
        usable = counts >= minimum_points

        if not np.any(usable):
            continue

        reference_filled = np.where(
            valid,
            reference_2d,
            0.0,
        )
        comparison_filled = np.where(
            valid,
            comparison_overlap,
            0.0,
        )

        reference_mean = np.zeros(
            comparison_overlap.shape[1],
            dtype=float,
        )
        comparison_mean = np.zeros_like(reference_mean)

        reference_mean[usable] = (
            np.sum(reference_filled[:, usable], axis=0)
            / counts[usable]
        )
        comparison_mean[usable] = (
            np.sum(comparison_filled[:, usable], axis=0)
            / counts[usable]
        )

        reference_centered = np.where(
            valid,
            reference_2d - reference_mean[np.newaxis, :],
            0.0,
        )
        comparison_centered = np.where(
            valid,
            comparison_overlap - comparison_mean[np.newaxis, :],
            0.0,
        )

        numerator = np.sum(
            reference_centered * comparison_centered,
            axis=0,
        )
        denominator = np.sqrt(
            np.sum(reference_centered ** 2, axis=0)
            * np.sum(comparison_centered ** 2, axis=0)
        )

        good = (
            usable
            & np.isfinite(denominator)
            & (denominator > 0.0)
        )

        correlation_values = np.full(
            comparison_overlap.shape[1],
            np.nan,
            dtype=float,
        )
        correlation_values[good] = np.clip(
            numerator[good] / denominator[good],
            -1.0,
            1.0,
        )

        correlation_map[:, shift_i] = correlation_values

    return correlation_map, selected_distances, reference_index


def find_best_shifts(correlation_map, shifts_minutes):
    """Find the maximum-correlation shift at every distance."""
    correlation_map = np.asarray(correlation_map, dtype=float)
    shifts_minutes = np.asarray(shifts_minutes, dtype=float)

    best_shift_minutes = np.full(
        correlation_map.shape[0],
        np.nan,
        dtype=float,
    )
    maximum_correlation = np.full(
        correlation_map.shape[0],
        np.nan,
        dtype=float,
    )

    for row_i, row in enumerate(correlation_map):
        finite_indices = np.flatnonzero(np.isfinite(row))
        if len(finite_indices) == 0:
            continue

        local_i = int(np.argmax(row[finite_indices]))
        full_i = int(finite_indices[local_i])

        best_shift_minutes[row_i] = shifts_minutes[full_i]
        maximum_correlation[row_i] = row[full_i]

    return best_shift_minutes, maximum_correlation


def fit_maximum_correlation_slope(
    distances_km,
    best_shift_minutes,
    maximum_correlation,
):
    """
    Fit shift_minutes = slope * distance_km + intercept, with the same
    continuity and robust-residual cleanup used by the separate analysis
    script.
    """
    distances_km = np.asarray(distances_km, dtype=float)
    best_shift_minutes = np.asarray(best_shift_minutes, dtype=float)
    maximum_correlation = np.asarray(maximum_correlation, dtype=float)

    candidate_mask = (
        np.isfinite(distances_km)
        & np.isfinite(best_shift_minutes)
        & np.isfinite(maximum_correlation)
        & (
            maximum_correlation
            >= ANALYSIS_MIN_CORRELATION_FOR_SLOPE
        )
        & (distances_km >= ANALYSIS_DISTANCE_MIN_KM)
        & (distances_km <= ANALYSIS_DISTANCE_MAX_KM)
    )

    retained_mask = candidate_mask.copy()

    if (
        ANALYSIS_REMOVE_RIDGE_OUTLIERS
        and np.count_nonzero(candidate_mask) >= 2
    ):
        candidate_indices = np.flatnonzero(candidate_mask)
        sorted_indices = candidate_indices[
            np.argsort(distances_km[candidate_indices])
        ]

        continuity_mask = np.zeros_like(candidate_mask, dtype=bool)
        first_index = int(sorted_indices[0])
        continuity_mask[first_index] = True
        previous_shift = best_shift_minutes[first_index]

        for index in sorted_indices[1:]:
            current_shift = best_shift_minutes[index]
            jump = abs(current_shift - previous_shift)

            if jump <= ANALYSIS_RIDGE_MAXIMUM_JUMP_MIN:
                continuity_mask[index] = True
                previous_shift = current_shift

        if (
            np.count_nonzero(continuity_mask)
            >= max(2, ANALYSIS_RIDGE_OUTLIER_MIN_POINTS)
        ):
            retained_mask = continuity_mask

        for _ in range(ANALYSIS_RIDGE_OUTLIER_MAX_ITERATIONS):
            if (
                np.count_nonzero(retained_mask)
                < max(3, ANALYSIS_RIDGE_OUTLIER_MIN_POINTS)
            ):
                break

            x = distances_km[retained_mask]
            y = best_shift_minutes[retained_mask]

            slope, intercept = np.polyfit(x, y, 1)
            residuals = y - (slope * x + intercept)
            median_residual = np.median(residuals)
            mad = np.median(
                np.abs(residuals - median_residual)
            )

            if not np.isfinite(mad) or np.isclose(mad, 0.0):
                break

            robust_sigma = 1.4826 * mad
            keep_local = (
                np.abs(residuals - median_residual)
                <= ANALYSIS_RIDGE_OUTLIER_SIGMA * robust_sigma
            )

            if np.all(keep_local):
                break

            retained_indices = np.flatnonzero(retained_mask)
            proposed = retained_mask.copy()
            proposed[retained_indices[~keep_local]] = False

            if (
                np.count_nonzero(proposed)
                < max(2, ANALYSIS_RIDGE_OUTLIER_MIN_POINTS)
            ):
                break

            retained_mask = proposed

    outlier_mask = candidate_mask & ~retained_mask
    number_of_points = int(np.count_nonzero(retained_mask))

    if number_of_points < 2:
        return {
            "slope_min_per_km": np.nan,
            "intercept_minutes": np.nan,
            "speed_km_per_min": np.nan,
            "speed_m_per_s": np.nan,
            "r_squared": np.nan,
            "number_of_fit_points": number_of_points,
            "number_of_outliers_removed": int(
                np.count_nonzero(outlier_mask)
            ),
            "fit_mask": retained_mask,
            "outlier_mask": outlier_mask,
        }

    x = distances_km[retained_mask]
    y = best_shift_minutes[retained_mask]

    slope, intercept = np.polyfit(x, y, 1)
    predicted = slope * x + intercept

    ss_res = np.sum((y - predicted) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)

    if ss_tot > 0.0:
        r_squared = 1.0 - ss_res / ss_tot
    else:
        r_squared = np.nan

    if np.isclose(slope, 0.0):
        speed_km_per_min = np.nan
        speed_m_per_s = np.nan
    else:
        # Sign convention:
        # calculate_shifted_correlation() defines a positive time shift as
        # moving the comparison row toward later times. Therefore propagation
        # toward increasing distance produces a negative ridge slope.
        # Flip the reciprocal so positive phase speed means propagation toward
        # increasing distance along the keogram cut.
        speed_km_per_min = -1.0 / slope
        speed_m_per_s = speed_km_per_min * 1000.0 / 60.0

    return {
        "slope_min_per_km": float(slope),
        "intercept_minutes": float(intercept),
        "speed_km_per_min": float(speed_km_per_min),
        "speed_m_per_s": float(speed_m_per_s),
        "r_squared": float(r_squared),
        "number_of_fit_points": number_of_points,
        "number_of_outliers_removed": int(
            np.count_nonzero(outlier_mask)
        ),
        "fit_mask": retained_mask,
        "outlier_mask": outlier_mask,
    }


def find_side_peak(
    correlation_profile,
    shifts_minutes,
    side,
):
    """Return the strongest finite non-zero-shift peak on one side."""
    correlation_profile = np.asarray(correlation_profile, dtype=float)
    shifts_minutes = np.asarray(shifts_minutes, dtype=float)

    if side == "left":
        mask = shifts_minutes < 0.0
    elif side == "right":
        mask = shifts_minutes > 0.0
    else:
        raise ValueError("side must be 'left' or 'right'.")

    indices = np.flatnonzero(
        mask & np.isfinite(correlation_profile)
    )

    if len(indices) == 0:
        return np.nan, np.nan

    best_index = int(
        indices[np.argmax(correlation_profile[indices])]
    )

    return (
        float(shifts_minutes[best_index]),
        float(correlation_profile[best_index]),
    )


def plot_one_analysis_angle(
    angle_deg,
    full_keogram,
    full_times_min,
    distance_km,
    processed_keogram,
    analysis_times_min,
    correlation_map,
    shifts_minutes,
    selected_distances,
    best_shift_minutes,
    maximum_correlation,
    slope_results,
    zero_profile,
    zero_profile_shifts_minutes,
    variant_label,
    output_file,
):
    """Save a five-panel diagnostic plot for one angle and one variant."""
    fig, axes = plt.subplots(
        5,
        1,
        figsize=(11, 19),
        gridspec_kw={
            "height_ratios": [1.45, 1.45, 1.05, 2.35, 1.05]
        },
    )

    (
        ax_full,
        ax_limited,
        ax_profiles,
        ax_map,
        ax_zero,
    ) = axes

    finite_abs = np.abs(full_keogram[np.isfinite(full_keogram)])
    if finite_abs.size > 0:
        keogram_limit = float(
            np.nanpercentile(finite_abs, 99.0)
        )
    else:
        keogram_limit = 1.0
    if not np.isfinite(keogram_limit) or keogram_limit <= 0.0:
        keogram_limit = 1.0

    mesh_full = ax_full.pcolormesh(
        full_times_min,
        distance_km,
        full_keogram.T,
        shading="auto",
        cmap=CMAP,
        vmin=-keogram_limit,
        vmax=keogram_limit,
    )
    fig.colorbar(
        mesh_full,
        ax=ax_full,
        fraction=0.025,
        pad=0.02,
        label="Keogram value",
    )
    ax_full.axhline(0.0, color="black", linestyle="--", linewidth=0.9)
    ax_full.set_title(
        f"{angle_deg}° full-range keogram\n{variant_label}"
    )
    ax_full.set_xlabel("Time (min)")
    ax_full.set_ylabel("Distance along slice (km)")

    selected_mask = (
        (distance_km >= ANALYSIS_DISTANCE_MIN_KM)
        & (distance_km <= ANALYSIS_DISTANCE_MAX_KM)
    )
    selected_keogram = processed_keogram[:, selected_mask]

    mesh_limited = ax_limited.pcolormesh(
        analysis_times_min,
        distance_km[selected_mask],
        selected_keogram.T,
        shading="auto",
        cmap=CMAP,
    )
    fig.colorbar(
        mesh_limited,
        ax=ax_limited,
        fraction=0.025,
        pad=0.02,
        label="Keogram value",
    )
    ax_limited.axhline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=0.9,
    )
    ax_limited.set_xlim(
        ANALYSIS_TIME_MIN,
        ANALYSIS_TIME_MAX,
    )
    ax_limited.set_ylim(
        ANALYSIS_DISTANCE_MIN_KM,
        ANALYSIS_DISTANCE_MAX_KM,
    )
    ax_limited.set_title(
        "Limited-range processed keogram"
        + (
            " (2-D smoothed)"
            if ANALYSIS_APPLY_2D_SMOOTHING
            else " (unsmoothed)"
        )
    )
    ax_limited.set_xlabel("Time (min)")
    ax_limited.set_ylabel("Distance along slice (km)")

    for target_distance in [0.0, 125.0, 250.0]:
        idx = int(
            np.argmin(np.abs(distance_km - target_distance))
        )
        ax_profiles.plot(
            analysis_times_min,
            processed_keogram[:, idx],
            linewidth=1.2,
            label=f"{distance_km[idx]:g} km",
        )
    ax_profiles.axhline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=0.8,
    )
    ax_profiles.set_xlim(
        ANALYSIS_TIME_MIN,
        ANALYSIS_TIME_MAX,
    )
    ax_profiles.set_title(
        "Processed profiles at 0, 125, and 250 km"
    )
    ax_profiles.set_xlabel("Time (min)")
    ax_profiles.set_ylabel("Value")
    ax_profiles.grid(True, alpha=0.3)
    ax_profiles.legend(loc="best", fontsize=8)

    if len(shifts_minutes) > 1:
        shift_spacing = float(np.median(np.diff(shifts_minutes)))
    else:
        shift_spacing = 1.0
    if len(selected_distances) > 1:
        distance_spacing = float(
            np.median(np.diff(selected_distances))
        )
    else:
        distance_spacing = 1.0

    image = ax_map.imshow(
        correlation_map,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        cmap=ANALYSIS_CORRELATION_CMAP,
        vmin=ANALYSIS_CORRELATION_VMIN,
        vmax=ANALYSIS_CORRELATION_VMAX,
        extent=[
            shifts_minutes[0] - shift_spacing / 2.0,
            shifts_minutes[-1] + shift_spacing / 2.0,
            selected_distances[0] - distance_spacing / 2.0,
            selected_distances[-1] + distance_spacing / 2.0,
        ],
    )
    fig.colorbar(
        image,
        ax=ax_map,
        fraction=0.025,
        pad=0.02,
        label="Pearson correlation with 0 km row",
    )
    ax_map.axvline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.0,
    )

    ridge_mask = (
        np.isfinite(best_shift_minutes)
        & np.isfinite(maximum_correlation)
        & (
            maximum_correlation
            >= ANALYSIS_MIN_CORRELATION_FOR_SLOPE
        )
    )
    if np.any(ridge_mask):
        ax_map.plot(
            best_shift_minutes[ridge_mask],
            selected_distances[ridge_mask],
            color="black",
            linewidth=1.1,
            label="Maximum correlation",
        )
        ax_map.scatter(
            best_shift_minutes[ridge_mask],
            selected_distances[ridge_mask],
            color="black",
            s=8,
        )

    outlier_mask = slope_results["outlier_mask"]
    if np.any(outlier_mask):
        ax_map.scatter(
            best_shift_minutes[outlier_mask],
            selected_distances[outlier_mask],
            marker="x",
            s=35,
            linewidth=1.1,
            label="Rejected ridge point",
        )

    slope = slope_results["slope_min_per_km"]
    intercept = slope_results["intercept_minutes"]

    if np.isfinite(slope):
        fit_distances = np.linspace(
            ANALYSIS_DISTANCE_MIN_KM,
            ANALYSIS_DISTANCE_MAX_KM,
            300,
        )
        fit_shifts = slope * fit_distances + intercept
        ax_map.plot(
            fit_shifts,
            fit_distances,
            linestyle="--",
            linewidth=2.0,
            label="Linear ridge fit",
        )

        ax_map.text(
            0.02,
            0.03,
            (
                f"Shift slope = {slope:.5f} min/km\n"
                f"Speed = {slope_results['speed_m_per_s']:.2f} m/s\n"
                f"$R^2$ = {slope_results['r_squared']:.3f}\n"
                f"Points = {slope_results['number_of_fit_points']}\n"
                f"Outliers removed = "
                f"{slope_results['number_of_outliers_removed']}"
            ),
            transform=ax_map.transAxes,
            ha="left",
            va="bottom",
            fontsize=9,
            bbox={
                "facecolor": "white",
                "edgecolor": "black",
                "alpha": 0.85,
            },
        )

    ax_map.set_xlim(
        ANALYSIS_SHIFT_MIN_MIN,
        ANALYSIS_SHIFT_MAX_MIN,
    )
    ax_map.set_ylim(
        ANALYSIS_DISTANCE_MIN_KM,
        ANALYSIS_DISTANCE_MAX_KM,
    )
    ax_map.set_title(
        f"{angle_deg}° shifted row correlation"
    )
    ax_map.set_xlabel("Shift applied to row (min)")
    ax_map.set_ylabel("Distance along slice (km)")
    ax_map.legend(loc="upper right", fontsize=8)

    left_shift, left_r = find_side_peak(
        zero_profile,
        zero_profile_shifts_minutes,
        side="left",
    )
    right_shift, right_r = find_side_peak(
        zero_profile,
        zero_profile_shifts_minutes,
        side="right",
    )

    ax_zero.plot(
        zero_profile_shifts_minutes,
        zero_profile,
        color="black",
        linewidth=1.3,
    )
    ax_zero.axvline(
        0.0,
        color="gray",
        linestyle="--",
        linewidth=0.9,
    )

    if np.isfinite(left_shift):
        ax_zero.scatter(
            [left_shift],
            [left_r],
            s=45,
            label=f"Left max: {left_shift:.1f} min",
        )
    if np.isfinite(right_shift):
        ax_zero.scatter(
            [right_shift],
            [right_r],
            s=45,
            marker="s",
            label=f"Right max: {right_shift:.1f} min",
        )

    ax_zero.set_xlim(
        ANALYSIS_ZERO_KM_SHIFT_MIN_MIN,
        ANALYSIS_ZERO_KM_SHIFT_MAX_MIN,
    )
    ax_zero.set_title("0 km autocorrelation versus shift")
    ax_zero.set_xlabel("Shift (min)")
    ax_zero.set_ylabel("Correlation")
    ax_zero.grid(True, alpha=0.3)
    ax_zero.legend(loc="best", fontsize=8)

    fig.suptitle(
        f"Correlation analysis: {variant_label}",
        fontsize=14,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.98])

    fig.savefig(
        output_file,
        dpi=ANALYSIS_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


# ==================================================
# Build all-angle analysis keograms
# ==================================================
def build_analysis_keograms():
    """
    Generate directional keograms for all analysis angles for:
      1. Wave + bias
      2. Wave + bias - calculated bias
      3. Wave only

    Returns
    -------
    distance_km : [distance]
    wave_bias_keograms : [angle, time, distance]
    corrected_keograms : [angle, time, distance]
    wave_only_keograms : [angle, time, distance]
    """
    n_angles = len(ANALYSIS_ANGLES_DEG)

    # Determine distance-axis length from one test cut.
    test_wave, test_bias, test_combined = make_three_frames(
        0,
        X_SMALL,
        Y_SMALL,
    )
    distance_km, test_cut = extract_directional_cut_any_angle(
        test_combined,
        ANALYSIS_ANGLES_DEG[0],
        width_pixels=KEOGRAM_WIDTH_PIXELS,
    )

    n_distance = len(test_cut)

    wave_bias_keograms = np.full(
        (n_angles, NT, n_distance),
        np.nan,
        dtype=np.float32,
    )
    corrected_keograms = np.full_like(
        wave_bias_keograms,
        np.nan,
    )
    wave_only_keograms = np.full_like(
        wave_bias_keograms,
        np.nan,
    )

    print(
        f"\nBuilding correlation-analysis keograms for "
        f"{n_angles} angles..."
    )

    for frame_index in range(NT):
        wave, bias, combined = make_three_frames(
            frame_index,
            X_SMALL,
            Y_SMALL,
        )
        calculated_bias = make_calculated_bias_frame(
            frame_index,
            X_SMALL,
            Y_SMALL,
        )

        have_calculated_bias = (
            np.isfinite(calc_bias_x_per_100km[frame_index])
            and np.isfinite(calc_bias_y_per_100km[frame_index])
        )

        if have_calculated_bias:
            corrected = combined - calculated_bias
        else:
            corrected = None

        for angle_index, angle_deg in enumerate(
            ANALYSIS_ANGLES_DEG
        ):
            _, wave_bias_cut = extract_directional_cut_any_angle(
                combined,
                angle_deg,
                width_pixels=KEOGRAM_WIDTH_PIXELS,
            )
            _, wave_only_cut = extract_directional_cut_any_angle(
                wave,
                angle_deg,
                width_pixels=KEOGRAM_WIDTH_PIXELS,
            )

            wave_bias_keograms[
                angle_index,
                frame_index,
                :,
            ] = wave_bias_cut

            wave_only_keograms[
                angle_index,
                frame_index,
                :,
            ] = wave_only_cut

            if corrected is not None:
                _, corrected_cut = extract_directional_cut_any_angle(
                    corrected,
                    angle_deg,
                    width_pixels=KEOGRAM_WIDTH_PIXELS,
                )
                corrected_keograms[
                    angle_index,
                    frame_index,
                    :,
                ] = corrected_cut

    return (
        distance_km,
        wave_bias_keograms,
        corrected_keograms,
        wave_only_keograms,
    )


def run_correlation_analysis():
    """Run and save the requested three-variant all-angle analysis."""
    (
        analysis_distance_km,
        wave_bias_keograms,
        corrected_keograms,
        wave_only_keograms,
    ) = build_analysis_keograms()

    analysis_time_mask = (
        (times_min >= ANALYSIS_TIME_MIN)
        & (times_min <= ANALYSIS_TIME_MAX)
    )
    analysis_times_min = times_min[analysis_time_mask]

    if (
        np.count_nonzero(analysis_time_mask)
        < ANALYSIS_MIN_VALID_POINTS
    ):
        raise ValueError(
            "The analysis interval contains too few time samples."
        )

    minutes_per_pixel = float(
        np.median(np.diff(times_min))
    )

    shift_min_pixels = int(
        np.ceil(
            ANALYSIS_SHIFT_MIN_MIN / minutes_per_pixel
        )
    )
    shift_max_pixels = int(
        np.floor(
            ANALYSIS_SHIFT_MAX_MIN / minutes_per_pixel
        )
    )
    shifts_pixels = np.arange(
        shift_min_pixels,
        shift_max_pixels + 1,
        dtype=int,
    )
    shifts_minutes = shifts_pixels * minutes_per_pixel

    zero_shift_min_pixels = int(
        np.ceil(
            ANALYSIS_ZERO_KM_SHIFT_MIN_MIN
            / minutes_per_pixel
        )
    )
    zero_shift_max_pixels = int(
        np.floor(
            ANALYSIS_ZERO_KM_SHIFT_MAX_MIN
            / minutes_per_pixel
        )
    )
    zero_shifts_pixels = np.arange(
        zero_shift_min_pixels,
        zero_shift_max_pixels + 1,
        dtype=int,
    )
    zero_shifts_minutes = (
        zero_shifts_pixels * minutes_per_pixel
    )

    selected_distance_mask = (
        (analysis_distance_km >= ANALYSIS_DISTANCE_MIN_KM)
        & (analysis_distance_km <= ANALYSIS_DISTANCE_MAX_KM)
    )

    variant_specs = [
        {
            "folder": "01_wave_plus_bias",
            "label": "Wave + bias",
            "keograms": wave_bias_keograms,
        },
        {
            "folder": (
                "02_wave_plus_bias_minus_calculated_bias"
            ),
            "label": "Wave + bias - calculated bias",
            "keograms": corrected_keograms,
        },
        {
            "folder": "03_wave_only",
            "label": "Wave only",
            "keograms": wave_only_keograms,
        },
    ]

    all_results = {}

    np.save(
        ANALYSIS_OUT_DIR / "angles_deg.npy",
        ANALYSIS_ANGLES_DEG,
    )
    np.save(
        ANALYSIS_OUT_DIR / "distance_km.npy",
        analysis_distance_km,
    )
    np.save(
        ANALYSIS_OUT_DIR / "analysis_times_min.npy",
        analysis_times_min,
    )
    np.save(
        ANALYSIS_OUT_DIR / "shifts_minutes.npy",
        shifts_minutes,
    )

    for variant in variant_specs:
        variant_out_dir = ANALYSIS_OUT_DIR / variant["folder"]
        plot_dir = variant_out_dir / "plots"
        array_dir = variant_out_dir / "arrays"

        plot_dir.mkdir(parents=True, exist_ok=True)
        array_dir.mkdir(parents=True, exist_ok=True)

        result_rows = []

        print("\n" + "#" * 72)
        print(f"Correlation analysis: {variant['label']}")
        print("#" * 72)

        for angle_index, angle_deg in enumerate(
            ANALYSIS_ANGLES_DEG
        ):
            full_keogram = np.asarray(
                variant["keograms"][angle_index],
                dtype=float,
            )

            limited_keogram = full_keogram[
                analysis_time_mask,
                :,
            ]

            # For the corrected case, finite values naturally exist only
            # over the calculated-bias interval; the shared time mask keeps
            # both variants identical in duration.
            if ANALYSIS_APPLY_2D_SMOOTHING:
                processed_keogram = smooth_keogram_2d_nanaware(
                    limited_keogram,
                    ANALYSIS_SMOOTH_SIGMA_TIME_PIXELS,
                    ANALYSIS_SMOOTH_SIGMA_DISTANCE_PIXELS,
                )
            else:
                processed_keogram = limited_keogram.copy()

            (
                correlation_map,
                selected_distances,
                reference_index,
            ) = calculate_correlation_map(
                processed_keogram,
                analysis_distance_km,
                ANALYSIS_REFERENCE_DISTANCE_KM,
                selected_distance_mask,
                shifts_pixels,
                minimum_points=ANALYSIS_MIN_VALID_POINTS,
            )

            (
                best_shift_minutes,
                maximum_correlation,
            ) = find_best_shifts(
                correlation_map,
                shifts_minutes,
            )

            slope_results = fit_maximum_correlation_slope(
                selected_distances,
                best_shift_minutes,
                maximum_correlation,
            )

            zero_index = int(
                np.argmin(
                    np.abs(
                        analysis_distance_km
                        - ANALYSIS_REFERENCE_DISTANCE_KM
                    )
                )
            )
            zero_series = processed_keogram[:, zero_index]

            zero_profile = np.asarray(
                [
                    calculate_shifted_correlation(
                        zero_series,
                        zero_series,
                        int(shift_pixels),
                        minimum_points=ANALYSIS_MIN_VALID_POINTS,
                    )
                    for shift_pixels in zero_shifts_pixels
                ],
                dtype=float,
            )

            angle_tag = f"{int(angle_deg):03d}deg"
            angle_array_dir = array_dir / angle_tag
            angle_array_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

            np.save(
                angle_array_dir / "processed_keogram.npy",
                processed_keogram,
            )
            np.save(
                angle_array_dir / "correlation_map.npy",
                correlation_map,
            )
            np.save(
                angle_array_dir / "best_shift_minutes.npy",
                best_shift_minutes,
            )
            np.save(
                angle_array_dir / "maximum_correlation.npy",
                maximum_correlation,
            )
            np.save(
                angle_array_dir / "selected_distances_km.npy",
                selected_distances,
            )
            np.save(
                angle_array_dir / "zero_km_correlation_profile.npy",
                zero_profile,
            )
            np.save(
                angle_array_dir / "zero_km_shifts_minutes.npy",
                zero_shifts_minutes,
            )

            np.savetxt(
                angle_array_dir / "ridge.csv",
                np.column_stack(
                    [
                        selected_distances,
                        best_shift_minutes,
                        maximum_correlation,
                        slope_results["fit_mask"].astype(int),
                        slope_results["outlier_mask"].astype(int),
                    ]
                ),
                delimiter=",",
                header=(
                    "distance_km,best_shift_minutes,"
                    "maximum_correlation,used_in_fit,"
                    "rejected_outlier"
                ),
                comments="",
            )

            plot_one_analysis_angle(
                angle_deg=int(angle_deg),
                full_keogram=full_keogram,
                full_times_min=times_min,
                distance_km=analysis_distance_km,
                processed_keogram=processed_keogram,
                analysis_times_min=analysis_times_min,
                correlation_map=correlation_map,
                shifts_minutes=shifts_minutes,
                selected_distances=selected_distances,
                best_shift_minutes=best_shift_minutes,
                maximum_correlation=maximum_correlation,
                slope_results=slope_results,
                zero_profile=zero_profile,
                zero_profile_shifts_minutes=zero_shifts_minutes,
                variant_label=variant["label"],
                output_file=(
                    plot_dir
                    / f"correlation_{angle_tag}.png"
                ),
            )

            result_rows.append(
                [
                    float(angle_deg),
                    slope_results["slope_min_per_km"],
                    slope_results["intercept_minutes"],
                    slope_results["speed_km_per_min"],
                    slope_results["speed_m_per_s"],
                    slope_results["r_squared"],
                    slope_results["number_of_fit_points"],
                    slope_results["number_of_outliers_removed"],
                ]
            )

            print(
                f"{angle_deg:3d}°: "
                f"speed={slope_results['speed_m_per_s']:+.2f} m/s, "
                f"R²={slope_results['r_squared']:.3f}, "
                f"points={slope_results['number_of_fit_points']}"
            )

        results = np.asarray(result_rows, dtype=float)
        all_results[variant["folder"]] = results

        np.savetxt(
            variant_out_dir / "all_angle_slope_summary.csv",
            results,
            delimiter=",",
            header=(
                "angle_deg,slope_min_per_km,intercept_minutes,"
                "speed_km_per_min,speed_m_per_s,r_squared,"
                "number_of_fit_points,number_of_outliers_removed"
            ),
            comments="",
        )
        np.save(
            variant_out_dir / "all_angle_slope_summary.npy",
            results,
        )

        fig, ax = plt.subplots(figsize=(10, 5.5))
        ax.plot(
            results[:, 0],
            results[:, 4],
            marker="o",
            linewidth=1.4,
        )
        ax.set_xlim(
            ANALYSIS_ANGLES_DEG.min(),
            ANALYSIS_ANGLES_DEG.max(),
        )
        ax.set_xlabel("Keogram angle (degrees)")
        ax.set_ylabel("Equivalent signed speed (m/s)")
        ax.set_title(
            f"Phase speed versus angle\n{variant['label']}"
        )
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        fig.savefig(
            variant_out_dir / "phase_speed_vs_angle.png",
            dpi=ANALYSIS_DPI,
            bbox_inches="tight",
        )
        plt.close(fig)

    # ==================================================
    # Combined phase-speed comparison
    # ==================================================
    fig, ax = plt.subplots(figsize=(11, 6))

    for variant in variant_specs:
        results = all_results[variant["folder"]]
        ax.plot(
            results[:, 0],
            results[:, 4],
            marker="o",
            linewidth=1.4,
            label=variant["label"],
        )

    # For this synthetic case, the true minimum apparent speed is the
    # magnitude of the prescribed phase speed.
    ax.axhline(
        abs(WAVE_SPEED_M_S),
        color="black",
        linestyle="--",
        linewidth=1.0,
        label=f"|input wave speed| = {abs(WAVE_SPEED_M_S):g} m/s",
    )
    ax.set_xlim(
        ANALYSIS_ANGLES_DEG.min(),
        ANALYSIS_ANGLES_DEG.max(),
    )
    ax.set_xticks(
        np.arange(
            ANALYSIS_ANGLES_DEG.min(),
            ANALYSIS_ANGLES_DEG.max() + 1,
            10,
        )
    )
    ax.set_xlabel("Keogram angle (degrees)")
    ax.set_ylabel("Equivalent signed speed (m/s)")
    ax.set_title(
        "Phase speed versus angle: bias correction comparison\n"
        f"Correlation interval: "
        f"{ANALYSIS_TIME_MIN:g}–{ANALYSIS_TIME_MAX:g} min"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=9)
    fig.tight_layout()
    phase_speed_comparison_path = (
        ANALYSIS_OUT_DIR
        / "phase_speed_vs_angle_comparison.png"
    )
    fig.savefig(
        phase_speed_comparison_path,
        dpi=ANALYSIS_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

    # ==================================================
    # Combined R^2 comparison
    # ==================================================
    fig, ax = plt.subplots(figsize=(11, 6))

    for variant in variant_specs:
        results = all_results[variant["folder"]]
        ax.plot(
            results[:, 0],
            results[:, 5],
            marker="o",
            linewidth=1.4,
            label=variant["label"],
        )

    ax.set_xlim(
        ANALYSIS_ANGLES_DEG.min(),
        ANALYSIS_ANGLES_DEG.max(),
    )
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel("Keogram angle (degrees)")
    ax.set_ylabel("R²")
    ax.set_title(
        "Correlation-ridge fit quality versus angle"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=9)
    fig.tight_layout()
    r2_comparison_path = (
        ANALYSIS_OUT_DIR
        / "r_squared_vs_angle_comparison.png"
    )
    fig.savefig(
        r2_comparison_path,
        dpi=ANALYSIS_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

    # ==================================================
    # Save a compact master summary
    # ==================================================
    master_rows = []
    for variant_index, variant in enumerate(
        variant_specs,
        start=1,
    ):
        results = all_results[variant["folder"]]
        for row in results:
            master_rows.append(
                [float(variant_index), *row.tolist()]
            )

    master_rows = np.asarray(master_rows, dtype=float)
    np.savetxt(
        ANALYSIS_OUT_DIR / "all_variants_all_angles_summary.csv",
        master_rows,
        delimiter=",",
        header=(
            "variant_index,angle_deg,slope_min_per_km,"
            "intercept_minutes,speed_km_per_min,speed_m_per_s,"
            "r_squared,number_of_fit_points,"
            "number_of_outliers_removed"
        ),
        comments="",
    )
    np.save(
        ANALYSIS_OUT_DIR / "all_variants_all_angles_summary.npy",
        master_rows,
    )

    with open(
        ANALYSIS_OUT_DIR / "analysis_parameters.txt",
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            f"Angles (deg): {ANALYSIS_ANGLES_DEG.tolist()}\n"
        )
        file.write(
            f"Analysis time range (min): "
            f"{ANALYSIS_TIME_MIN} to {ANALYSIS_TIME_MAX}\n"
        )
        file.write(
            f"Distance range (km): "
            f"{ANALYSIS_DISTANCE_MIN_KM} to "
            f"{ANALYSIS_DISTANCE_MAX_KM}\n"
        )
        file.write(
            f"Correlation shift range (min): "
            f"{ANALYSIS_SHIFT_MIN_MIN} to "
            f"{ANALYSIS_SHIFT_MAX_MIN}\n"
        )
        file.write(
            f"2-D smoothing: {ANALYSIS_APPLY_2D_SMOOTHING}\n"
        )
        file.write(
            f"Smoothing sigma time/distance (pixels): "
            f"{ANALYSIS_SMOOTH_SIGMA_TIME_PIXELS}, "
            f"{ANALYSIS_SMOOTH_SIGMA_DISTANCE_PIXELS}\n"
        )
        file.write(
            "Variant 1: Wave + bias\n"
        )
        file.write(
            "Variant 2: Wave + bias - calculated bias\n"
        )
        file.write(
            "Variant 3: Wave only\n"
        )

    print("\nCorrelation analysis complete.")
    print("Analysis output:", ANALYSIS_OUT_DIR.resolve())

    return {
        "results": all_results,
        "phase_speed_plot": phase_speed_comparison_path,
        "r_squared_plot": r2_comparison_path,
    }


analysis_outputs = run_correlation_analysis()


# ==================================================
# Animation builder
# ==================================================
def build_animation(angle_index, keogram_angle_deg):
    """
    Build one full animation for one keogram/profile direction.
    """
    distance_axis = distances_by_angle[angle_index]
    angle_keograms = keograms[angle_index]
    angle_profiles = profiles[angle_index]
    angle_profile_distances = profile_actual_distances[angle_index]

    fig, axes = plt.subplots(
        4,
        3,
        figsize=(18.5, 15.5),
        gridspec_kw={
            "height_ratios": [1.25, 1.25, 0.95, 0.95],
            "hspace": 0.34,
            "wspace": 0.24,
        },
    )

    fig.subplots_adjust(
        top=0.94,
        bottom=0.06,
        left=0.055,
        right=0.91,
    )

    # --------------------------------------------------
    # Row 1: big FOV images
    # --------------------------------------------------
    initial_big_frames = make_three_frames(0, X_BIG, Y_BIG)

    top_vmax = [TOP_WAVE_VMAX, TOP_BIAS_VMAX, TOP_COMBINED_VMAX]
    top_images = []

    for col in range(3):
        ax = axes[0, col]

        image = ax.imshow(
            initial_big_frames[col],
            cmap=CMAP,
            vmin=-top_vmax[col],
            vmax=top_vmax[col],
            origin="upper",
            extent=[
                -BIG_FIELD_SIZE_KM / 2.0,
                BIG_FIELD_SIZE_KM / 2.0,
                BIG_FIELD_SIZE_KM / 2.0,
                -BIG_FIELD_SIZE_KM / 2.0,
            ],
            interpolation="nearest",
            aspect="equal",
        )
        top_images.append(image)

        # Box marking the central 500 x 500 km FOV.
        box = Rectangle(
            (
                -SMALL_FIELD_SIZE_KM / 2.0,
                -SMALL_FIELD_SIZE_KM / 2.0,
            ),
            SMALL_FIELD_SIZE_KM,
            SMALL_FIELD_SIZE_KM,
            fill=False,
            edgecolor="black",
            linewidth=2.0,
            zorder=5,
        )
        ax.add_patch(box)

        ax.set_title(f"Large FOV: {variant_names[col]}")
        ax.set_xlabel("x (km)")
        if col == 0:
            ax.set_ylabel("y (km; + downward)")

        cbar = fig.colorbar(
            image,
            ax=ax,
            fraction=0.046,
            pad=0.03,
        )
        cbar.set_label("Amplitude")

    # --------------------------------------------------
    # Row 2: small FOV images + slope arrows
    # --------------------------------------------------
    initial_small_frames = make_three_frames(0, X_SMALL, Y_SMALL)

    small_images = []
    slope_quivers = []
    slope_texts = []

    for col in range(3):
        ax = axes[1, col]

        image = ax.imshow(
            initial_small_frames[col],
            cmap=CMAP,
            vmin=-SMALL_COMMON_VMAX,
            vmax=SMALL_COMMON_VMAX,
            origin="upper",
            extent=[
                -SMALL_FIELD_SIZE_KM / 2.0,
                SMALL_FIELD_SIZE_KM / 2.0,
                SMALL_FIELD_SIZE_KM / 2.0,
                -SMALL_FIELD_SIZE_KM / 2.0,
            ],
            interpolation="nearest",
            aspect="equal",
        )
        small_images.append(image)

        sx0 = plane_slope_x[col, 0]
        sy0 = plane_slope_y[col, 0]
        mag0 = np.hypot(sx0, sy0)

        if mag0 > 0.0:
            arrow_length = MAX_ARROW_LENGTH_KM * mag0 / max_plane_slope
            arrow_u = arrow_length * sx0 / mag0
            arrow_v = arrow_length * sy0 / mag0
        else:
            arrow_u = 0.0
            arrow_v = 0.0

        quiver = ax.quiver(
            [0.0],
            [0.0],
            [arrow_u],
            [arrow_v],
            angles="xy",
            scale_units="xy",
            scale=1.0,
            width=0.010,
            color="black",
            zorder=6,
        )
        slope_quivers.append(quiver)

        slope_text = ax.text(
            0.02,
            0.98,
            "",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9,
            bbox=dict(
                facecolor="white",
                alpha=0.75,
                edgecolor="none",
            ),
            zorder=7,
        )
        slope_texts.append(slope_text)

        ax.set_title(f"500 x 500 km: {variant_names[col]}")
        ax.set_xlabel("x (km)")
        if col == 0:
            ax.set_ylabel("y (km; + downward)")

    small_cbar = fig.colorbar(
        small_images[-1],
        ax=axes[1, :].tolist(),
        fraction=0.018,
        pad=0.015,
    )
    small_cbar.set_label("Amplitude")

    # --------------------------------------------------
    # Row 3: directional keograms
    # --------------------------------------------------
    keogram_images = []
    keogram_time_lines = []

    for col in range(3):
        ax = axes[2, col]

        # imshow expects [distance, time], hence transpose.
        initial_partial = np.full_like(angle_keograms[col].T, np.nan)
        initial_partial[:, :1] = angle_keograms[col, :1].T

        image = ax.imshow(
            initial_partial,
            cmap=CMAP,
            vmin=-SMALL_COMMON_VMAX,
            vmax=SMALL_COMMON_VMAX,
            origin="lower",
            aspect="auto",
            extent=[
                times_min[0],
                times_min[-1],
                distance_axis[0],
                distance_axis[-1],
            ],
            interpolation="nearest",
        )
        keogram_images.append(image)

        current_line = ax.axvline(
            times_min[0],
            color="black",
            linewidth=1.3,
            alpha=0.9,
        )
        keogram_time_lines.append(current_line)

        for requested_distance in PROFILE_DISTANCES_KM:
            ax.axhline(
                requested_distance,
                color="0.35",
                linewidth=0.8,
                linestyle="--",
                alpha=0.7,
            )

        ax.set_title(
            f"{keogram_angle_deg:.0f}° keogram: {variant_names[col]}"
        )
        ax.set_xlim(times_min[0], times_min[-1])
        ax.set_ylim(distance_axis[0], distance_axis[-1])
        ax.set_xlabel("Time (min)")

        if col == 0:
            if np.isclose(keogram_angle_deg % 180.0, 0.0):
                ax.set_ylabel("x distance (km)")
            else:
                ax.set_ylabel("y distance (km; + south)")

    keo_cbar = fig.colorbar(
        keogram_images[-1],
        ax=axes[2, :].tolist(),
        fraction=0.018,
        pad=0.015,
    )
    keo_cbar.set_label("Amplitude")

    # --------------------------------------------------
    # Row 4: profiles at 0, 125, 250 km
    # --------------------------------------------------
    profile_lines = [
        [None] * len(PROFILE_DISTANCES_KM)
        for _ in range(3)
    ]
    profile_time_lines = []

    for col in range(3):
        ax = axes[3, col]

        for profile_index, actual_distance in enumerate(
            angle_profile_distances
        ):
            line, = ax.plot(
                times_min[:1],
                angle_profiles[col, profile_index, :1],
                linewidth=1.5,
                label=f"{actual_distance:.0f} km",
            )
            profile_lines[col][profile_index] = line

        current_line = ax.axvline(
            times_min[0],
            color="black",
            linewidth=1.2,
            alpha=0.8,
        )
        profile_time_lines.append(current_line)

        ax.axhline(
            0.0,
            color="0.5",
            linewidth=0.8,
            linestyle="--",
        )

        ax.set_title(
            f"{keogram_angle_deg:.0f}° profiles: {variant_names[col]}"
        )
        ax.set_xlim(times_min[0], times_min[-1])
        ax.set_ylim(*PROFILE_YLIM)
        ax.set_xlabel("Time (min)")

        if col == 0:
            ax.set_ylabel("Amplitude")

        ax.grid(True, alpha=0.25)
        ax.legend(
            loc="best",
            fontsize=8,
            ncol=3,
        )

    # ==================================================
    # Animation update
    # ==================================================
    def update(frame_index):
        big_frames = make_three_frames(
            frame_index,
            X_BIG,
            Y_BIG,
        )
        small_frames = make_three_frames(
            frame_index,
            X_SMALL,
            Y_SMALL,
        )

        current_time = times_min[frame_index]
        artists = []

        # Row 1
        for col in range(3):
            top_images[col].set_data(big_frames[col])
            artists.append(top_images[col])

        # Row 2: images + plane-fit slope arrows
        for col in range(3):
            small_images[col].set_data(small_frames[col])
            artists.append(small_images[col])

            sx = plane_slope_x[col, frame_index]
            sy = plane_slope_y[col, frame_index]

            magnitude = np.hypot(sx, sy)
            angle = np.degrees(np.arctan2(sy, sx))
            angle = (angle + 180.0) % 360.0 - 180.0

            if magnitude > 0.0:
                arrow_length = (
                    MAX_ARROW_LENGTH_KM
                    * magnitude
                    / max_plane_slope
                )
                arrow_u = arrow_length * sx / magnitude
                arrow_v = arrow_length * sy / magnitude
            else:
                arrow_u = 0.0
                arrow_v = 0.0

            slope_quivers[col].set_UVC(
                [arrow_u],
                [arrow_v],
            )
            artists.append(slope_quivers[col])

            slope_texts[col].set_text(
                f"x slope = {sx:+.4f} /100 km\n"
                f"y slope = {sy:+.4f} /100 km\n"
                f"|slope| = {magnitude:.4f} /100 km\n"
                f"direction = {angle:+.1f}°"
            )
            artists.append(slope_texts[col])

        # Row 3: accumulated keogram through current frame
        for col in range(3):
            partial_keogram = np.full_like(
                angle_keograms[col].T,
                np.nan,
            )
            partial_keogram[:, : frame_index + 1] = (
                angle_keograms[col, : frame_index + 1].T
            )

            keogram_images[col].set_data(partial_keogram)
            keogram_time_lines[col].set_xdata(
                [current_time, current_time]
            )

            artists.extend(
                [
                    keogram_images[col],
                    keogram_time_lines[col],
                ]
            )

        # Row 4: accumulated profiles through current frame
        for col in range(3):
            for profile_index in range(
                len(PROFILE_DISTANCES_KM)
            ):
                profile_lines[col][profile_index].set_data(
                    times_min[: frame_index + 1],
                    angle_profiles[
                        col,
                        profile_index,
                        : frame_index + 1,
                    ],
                )
                artists.append(
                    profile_lines[col][profile_index]
                )

            profile_time_lines[col].set_xdata(
                [current_time, current_time]
            )
            artists.append(profile_time_lines[col])

        bx = bias_x_per_100km[frame_index]
        by = bias_y_per_100km[frame_index]
        wave_angle = propagation_angles_deg[frame_index]

        fig.suptitle(
            "Synthetic wave + changing planar bias\n"
            f"Keogram/profile direction = "
            f"{keogram_angle_deg:.0f}°   |   "
            f"t = {current_time:.0f} min   |   "
            f"wave angle = {wave_angle:.1f}°   |   "
            f"bias = ({bx:+.4f}, {by:+.4f}) amp/100 km",
            fontsize=15,
        )

        return artists

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=NT,
        interval=1000.0 / FPS,
        blit=False,
        repeat=True,
    )

    return fig, ani


# ==================================================
# Save one animation per angle
# ==================================================
def save_animation_for_angle(fig, ani, angle_deg):
    angle_tag = f"{int(round(angle_deg)) % 180:03d}deg"
    preferred = PREFERRED_ANIMATION_FORMAT.lower()

    if (
        preferred == "mp4"
        and animation.writers.is_available("ffmpeg")
    ):
        output_path = (
            OUT_DIR
            / f"{OUTPUT_BASENAME}_keogram_{angle_tag}.mp4"
        )
        print("Saving MP4:", output_path)

        writer = animation.FFMpegWriter(
            fps=FPS,
            bitrate=3500,
            metadata={
                "artist": "synthetic wave/bias animation"
            },
        )
        ani.save(
            output_path,
            writer=writer,
            dpi=ANIMATION_DPI,
        )
    else:
        output_path = (
            OUT_DIR
            / f"{OUTPUT_BASENAME}_keogram_{angle_tag}.gif"
        )
        print(
            "ffmpeg unavailable or GIF requested; "
            "saving GIF:",
            output_path,
        )

        writer = animation.PillowWriter(fps=FPS)
        ani.save(
            output_path,
            writer=writer,
            dpi=ANIMATION_DPI,
        )

    plt.close(fig)
    return output_path


saved_animation_paths = []

for angle_index, keogram_angle_deg in enumerate(
    KEOGRAM_ANGLES_DEG
):
    print(
        f"\nBuilding animation for "
        f"{keogram_angle_deg:.0f}° keogram..."
    )

    fig, ani = build_animation(
        angle_index,
        keogram_angle_deg,
    )

    saved_path = save_animation_for_angle(
        fig,
        ani,
        keogram_angle_deg,
    )

    saved_animation_paths.append(saved_path)


# ==================================================
# Save parameter summary
# ==================================================
parameter_path = OUT_DIR / "animation_parameters.txt"

with open(parameter_path, "w", encoding="utf-8") as file:
    file.write(f"Number of frames: {NT}\n")
    file.write(
        f"Time range (min): "
        f"{times_min[0]} to {times_min[-1]}\n"
    )
    file.write(f"Time step (s): {DT_SECONDS}\n")

    file.write(
        f"Small FOV (km): "
        f"{SMALL_FIELD_SIZE_KM} x {SMALL_FIELD_SIZE_KM}\n"
    )
    file.write(
        f"Small resolution x/y (km): "
        f"{SMALL_DX_KM}, {SMALL_DY_KM}\n"
    )

    file.write(
        f"Big FOV (km): "
        f"{BIG_FIELD_SIZE_KM} x {BIG_FIELD_SIZE_KM}\n"
    )
    file.write(
        f"Big resolution x/y (km): "
        f"{BIG_DX_KM}, {BIG_DY_KM}\n"
    )

    file.write(f"Wavelength (km): {WAVELENGTH_KM}\n")
    file.write(f"Gaussian sigma (km): {GAUSSIAN_WIDTH_KM}\n")
    file.write(f"Maximum wave amplitude: {MAX_AMPLITUDE}\n")
    file.write(f"Wave speed (m/s): {WAVE_SPEED_M_S}\n")
    file.write(f"Wave time shift (min): {WAVE_TIME_SHIFT_MIN}\n")

    file.write(
        "Propagation angle schedule: "
        f"{PROPAGATION_ANGLE_START_DEG} deg at "
        f"{PROPAGATION_ANGLE_START_TIME_MIN} min to "
        f"{PROPAGATION_ANGLE_END_DEG} deg at "
        f"{PROPAGATION_ANGLE_END_TIME_MIN} min\n"
    )

    file.write(
        f"Bias x slope: {BIAS_X_START_PER_100KM} "
        f"at 0 min to {BIAS_X_END_PER_100KM} "
        f"at {BIAS_END_TIME_MIN} min, then held through "
        f"{TOTAL_TIME_MIN} min\n"
    )
    file.write(
        f"Bias y slope: {BIAS_Y_START_PER_100KM} "
        f"at 0 min to {BIAS_Y_END_PER_100KM} "
        f"at {BIAS_END_TIME_MIN} min, then held through "
        f"{TOTAL_TIME_MIN} min\n"
    )

    file.write(
        f"Keogram angles (deg): {KEOGRAM_ANGLES_DEG}\n"
    )
    file.write(
        f"Keogram width (pixels): "
        f"{KEOGRAM_WIDTH_PIXELS}\n"
    )
    file.write(
        f"Requested profile distances (km): "
        f"{PROFILE_DISTANCES_KM}\n"
    )

    for angle_index, angle_deg in enumerate(
        KEOGRAM_ANGLES_DEG
    ):
        file.write(
            f"Actual profile distances for "
            f"{angle_deg:.0f} deg (km): "
            f"{profile_actual_distances[angle_index].tolist()}\n"
        )

    file.write(
        f"Slope vector plot: {slope_plot_path.name}\n"
    )
    file.write(
        f"Calculated bias plot: {calculated_bias_plot_path.name}\n"
    )
    file.write(
        "Calculated bias anchor 1: "
        f"t={CALC_BIAS_FIRST_CENTER_MIN:.1f} min, "
        f"x={calc_bias_anchor_x[0]:+.8f}, "
        f"y={calc_bias_anchor_y[0]:+.8f} amp/100 km\n"
    )
    file.write(
        "Calculated bias anchor 2: "
        f"t={CALC_BIAS_SECOND_CENTER_MIN:.1f} min, "
        f"x={calc_bias_anchor_x[1]:+.8f}, "
        f"y={calc_bias_anchor_y[1]:+.8f} amp/100 km\n"
    )
    file.write(
        "Calculated-bias comparison panel 4: "
        "calculated bias - actual imposed bias\n"
    )


print("\nFinished.")
for path in saved_animation_paths:
    print("Animation:", path)

print("Slope-vector plot:", slope_plot_path)
print("Calculated-bias plot:", calculated_bias_plot_path)
print(
    "Diagnostics:",
    OUT_DIR / "animation_diagnostics.npz",
)
print("Parameters:", parameter_path)


Precomputing diagnostics for 151 frames...
Calculated-bias anchor at 60 min (average 0-120 min): x=-0.013070, y=+0.007603 amp/100 km
Calculated-bias anchor at 240 min (average 180-300 min): x=+0.022820, y=+0.005832 amp/100 km
Slope-vector plot: synthetic_wave_bias_animation_output/slope_vectors_over_time.png
Calculated-bias plot: synthetic_wave_bias_animation_output/calculated_bias_comparison.png

Building correlation-analysis keograms for 21 angles...

########################################################################
Correlation analysis: Wave + bias
########################################################################
 60°: speed=+670.49 m/s, R²=0.908, points=251
 65°: speed=+625.32 m/s, R²=0.923, points=251
 70°: speed=+588.84 m/s, R²=0.925, points=251
 75°: speed=+525.49 m/s, R²=0.940, points=251
 80°: speed=+491.22 m/s, R²=0.950, points=251
 85°: speed=+470.33 m/s, R²=0.953, points=251
 90°: speed=+435.94 m/s, R²=0.957, points=251
 95°: speed=+410.19 m/s, R²=0.963, point